# Deployable MedSAM Study Notebook

This notebook is the main study workflow for dataset inspection, prompt robustness analysis, calibration-region analysis, and later experiment stages. The current section focuses on Kvasir-SEG dataset preparation and prompt protocol validation. It is designed for a WSL virtual environment and does not download models or datasets.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

print('Python executable:', sys.executable)
print('Repository root:', REPO_ROOT)

missing = []
for package_name, import_name in [('Pillow', 'PIL'), ('numpy', 'numpy'), ('matplotlib', 'matplotlib'), ('pandas', 'pandas')]:
    try:
        __import__(import_name)
    except ImportError:
        missing.append(package_name)

if missing:
    print('Missing notebook package(s):', ', '.join(missing))
    print('Install inside your WSL venv with: python -m pip install -r requirements.txt')
else:
    print('Notebook dependencies available.')

In [ ]:
from deployable_medsam.data import (
    create_split_manifest,
    discover_segmentation_samples,
    load_binary_mask,
    load_rgb_image,
    mask_to_uint8_image,
    write_split_manifest,
)
from deployable_medsam.data.prompts import generate_noisy_box_prompt, mask_to_box_prompt
from deployable_medsam.quantization.calibration import (
    build_calibration_manifest,
    select_representative_samples,
    write_manifest,
)
from research_runtime.metrics import (
    binary_brier_score,
    binary_ece,
    boundary_mask_from_binary_masks,
    dice_score,
    foreground_mask,
    iou_score,
    precision_score,
    recall_score,
    roi_mask_from_binary_masks,
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle

print('Project imports loaded.')

## Dataset Path Setup

Put Kvasir-SEG images and masks under `data/kvasir_seg/images` and `data/kvasir_seg/masks`, or edit `DATASET_ROOT` below.

In [ ]:
DATASET_ROOT = REPO_ROOT / 'data' / 'kvasir_seg'
IMAGE_DIR = DATASET_ROOT / 'images'
MASK_DIR = DATASET_ROOT / 'masks'
INPUT_SIZE = (256, 256)
SEED = 1
JITTER_LEVELS = [0, 5, 10, 20]
EXPECTED_DATASET_LAYOUT = f"""Expected Kvasir-SEG local layout:
{DATASET_ROOT}/
  images/  # image files, for example .jpg or .png
  masks/   # matching mask files with the same filename stem

Download/extract Kvasir-SEG, then place the files into those two folders before running dataset-dependent cells.
"""

print('Dataset root:', DATASET_ROOT)
print('Image dir exists:', IMAGE_DIR.exists(), IMAGE_DIR)
print('Mask dir exists:', MASK_DIR.exists(), MASK_DIR)

## Dataset Discovery

In [ ]:
SUPPORTED_IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
image_files = [path for path in IMAGE_DIR.iterdir() if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS] if IMAGE_DIR.exists() else []
mask_files = [path for path in MASK_DIR.iterdir() if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS] if MASK_DIR.exists() else []
if not IMAGE_DIR.exists() or not MASK_DIR.exists() or not image_files or not mask_files:
    raise RuntimeError(EXPECTED_DATASET_LAYOUT)

samples = discover_segmentation_samples(IMAGE_DIR, MASK_DIR, dataset='kvasir_seg')
print(f'Discovered {len(samples)} image/mask pairs.')
print('First sample ids:', [sample.sample_id for sample in samples[:5]])
pd.DataFrame([sample.__dict__ for sample in samples[:10]])

## Dataset Split Manifest

In [ ]:
split_manifest = create_split_manifest(samples, dataset='kvasir_seg', seed=SEED)
split_path = REPO_ROOT / 'results' / 'splits' / 'kvasir_seg_seed1.json'
write_split_manifest(split_manifest, split_path)
print('Split counts:', split_manifest.counts)
print('Wrote:', split_path)
pd.DataFrame([
    {'split': split_name, 'sample_id': sample.sample_id, 'image_path': sample.image_path, 'mask_path': sample.mask_path}
    for split_name, split_samples in split_manifest.samples.items()
    for sample in split_samples[:5]
])

## Image, Mask, and Prompt Inspection

In [ ]:
inspection_samples = samples[: min(6, len(samples))]
fig, axes = plt.subplots(len(inspection_samples), 3, figsize=(12, 4 * len(inspection_samples)))
if len(inspection_samples) == 1:
    axes = np.array([axes])

for row_index, sample in enumerate(inspection_samples):
    image = load_rgb_image(sample.image_path, size=INPUT_SIZE)
    mask = load_binary_mask(sample.mask_path, size=INPUT_SIZE)
    mask_image = mask_to_uint8_image(mask)
    clean_box = mask_to_box_prompt(mask)
    noisy_box = generate_noisy_box_prompt(mask, jitter_pixels=10, seed=SEED + row_index)

    axes[row_index, 0].imshow(image)
    axes[row_index, 0].set_title(f'{sample.sample_id} image')
    axes[row_index, 1].imshow(mask_image, cmap='gray')
    axes[row_index, 1].set_title('binary mask')
    axes[row_index, 2].imshow(image)
    for box, color, label in [(clean_box, 'lime', 'clean'), (noisy_box, 'red', 'noisy 10px')]:
        x0, y0, x1, y1 = box.as_xyxy()
        axes[row_index, 2].add_patch(Rectangle((x0, y0), x1 - x0 + 1, y1 - y0 + 1, fill=False, edgecolor=color, linewidth=2, label=label))
    axes[row_index, 2].set_title('clean + noisy box prompts')
    axes[row_index, 2].legend(loc='lower right')
    for col in range(3):
        axes[row_index, col].axis('off')

plt.tight_layout()

## Prompt Robustness Inspection

In [ ]:
sample = samples[0]
mask = load_binary_mask(sample.mask_path, size=INPUT_SIZE)
prompt_records = []
for jitter in JITTER_LEVELS:
    box = generate_noisy_box_prompt(mask, jitter_pixels=jitter, seed=SEED)
    x0, y0, x1, y1 = box.as_xyxy()
    assert 0 <= x0 <= x1 < INPUT_SIZE[0]
    assert 0 <= y0 <= y1 < INPUT_SIZE[1]
    prompt_records.append({'sample_id': sample.sample_id, 'jitter_pixels': jitter, 'box_xyxy': box.as_xyxy()})

pd.DataFrame(prompt_records)

## Calibration Region Analysis

This synthetic tiny-lesion case shows why whole-image ECE is not enough for medical segmentation.

In [ ]:
calibration_target = [[0 for _ in range(8)] for _ in range(8)]
calibration_target[3][3] = 1
calibration_prediction = [[0 for _ in range(8)] for _ in range(8)]

foreground_region = foreground_mask(calibration_target)
calibration_roi = roi_mask_from_binary_masks(calibration_prediction, calibration_target, padding=1)
boundary_region = boundary_mask_from_binary_masks(calibration_prediction, calibration_target, radius=1)

calibration_metrics = {
    'dice': dice_score(calibration_prediction, calibration_target),
    'iou': iou_score(calibration_prediction, calibration_target),
    'precision': precision_score(calibration_prediction, calibration_target),
    'recall': recall_score(calibration_prediction, calibration_target),
    'brier_full': binary_brier_score(calibration_prediction, calibration_target),
    'brier_roi': binary_brier_score(calibration_prediction, calibration_target, sample_mask=calibration_roi),
    'binary_ece_full': binary_ece(calibration_prediction, calibration_target),
    'binary_ece_foreground': binary_ece(calibration_prediction, calibration_target, sample_mask=foreground_region),
    'binary_ece_roi': binary_ece(calibration_prediction, calibration_target, sample_mask=calibration_roi),
    'binary_ece_boundary': binary_ece(calibration_prediction, calibration_target, sample_mask=boundary_region),
}
pd.DataFrame([calibration_metrics])

## Calibration Cohort Manifest

In [ ]:
validation_samples = split_manifest.samples['validation']
calibration_samples = select_representative_samples(validation_samples, target_samples=min(256, len(validation_samples)))
calibration_manifest = build_calibration_manifest(
    samples=calibration_samples,
    dataset='kvasir_seg',
    split='validation',
    preprocessing_contract='Resize to 256x256, RGB image preprocessing, binary mask available for ROI/boundary evaluation.',
)
calibration_path = REPO_ROOT / 'results' / 'raw' / 'medsam_calibration_manifest.json'
write_manifest(calibration_manifest, calibration_path)
print(f'Selected {len(calibration_samples)} calibration samples.')
print('Wrote:', calibration_path)
pd.DataFrame([sample.__dict__ for sample in calibration_samples[:10]])

## Study Execution Controls

Use these cells to run project checks and teacher baseline jobs from inside the notebook. All execution toggles default to `False` so re-running the notebook does not accidentally start model inference.


In [ ]:
import subprocess
import sys
from datetime import datetime

def run_project_command(command, *, enabled=False, label=None, stdout_mode="print"):
    """Run a project command from the notebook and stream or summarize its output."""
    display_label = label or " ".join(str(part) for part in command)
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {display_label}")
    print("Command:", " ".join(str(part) for part in command))
    if not enabled:
        print("Skipped. Set the matching RUN_* variable to True to execute this cell.")
        return None

    completed = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True)
    if completed.stdout:
        if stdout_mode == "suppress":
            print(f"STDOUT suppressed ({len(completed.stdout)} characters).")
        else:
            print(completed.stdout)
    if completed.stderr:
        print("STDERR:")
        print(completed.stderr)
    completed.check_returncode()
    return completed

def python_script(script_name, *args):
    return [sys.executable, str(REPO_ROOT / script_name), *map(str, args)]

### Validation Checks

These checks do not download MedSAM. Notebook JSON validation suppresses stdout so it does not dump the full notebook into the cell output.


In [ ]:
RUN_VALIDATION_CHECKS = False

validation_commands = [
    {"command": [sys.executable, "-m", "compileall", "src", "scripts"], "label": "Compile source and scripts"},
    {"command": python_script("scripts/smoke_test_dataset_discovery.py"), "label": "Dataset discovery smoke test"},
    {"command": python_script("scripts/smoke_test_prompt_generation.py"), "label": "Prompt generation smoke test"},
    {"command": python_script("scripts/smoke_test_segmentation_metrics.py"), "label": "Segmentation metrics smoke test"},
    {"command": python_script("scripts/smoke_test_calibration_manifest.py"), "label": "Calibration manifest smoke test"},
    {"command": python_script("scripts/smoke_test_teacher_baseline_contract.py"), "label": "Teacher baseline contract smoke test"},
    {"command": python_script("scripts/smoke_test_teacher_baseline_outputs.py"), "label": "Teacher baseline output smoke test"},
    {
        "command": [sys.executable, "-m", "json.tool", str(REPO_ROOT / "notebooks" / "deployable_medsam_study.ipynb")],
        "label": "Notebook JSON validation",
        "stdout_mode": "suppress",
    },
]

for spec in validation_commands:
    run_project_command(spec["command"], enabled=RUN_VALIDATION_CHECKS, label=spec["label"], stdout_mode=spec.get("stdout_mode", "print"))

### Teacher Baseline Runs

These cells call the teacher baseline script from Jupyter with safe run tags. Use `--save-predictions` when you want visual error-review grids.


In [ ]:
RUN_TEACHER_PILOT = False
RUN_TEACHER_VALIDATION_CLEAN = False
RUN_TEACHER_VALIDATION_PROMPT_SENSITIVITY = False
RUN_TEACHER_TEST_PROMPT_SENSITIVITY = False
RUN_TEACHER_TRAIN_DISTILLATION_CLEAN = False

teacher_run_specs = [
    {
        "enabled": RUN_TEACHER_PILOT,
        "label": "Teacher baseline pilot: validation, 2 samples, clean prompts",
        "command": python_script("scripts/run_teacher_baseline.py", "--split", "validation", "--sample-limit", 2, "--prompt-mode", "clean", "--run-tag", "pilot_clean_n2", "--save-predictions"),
    },
    {
        "enabled": RUN_TEACHER_VALIDATION_CLEAN,
        "label": "Teacher baseline: full validation, clean prompts",
        "command": python_script("scripts/run_teacher_baseline.py", "--split", "validation", "--prompt-mode", "clean", "--run-tag", "validation_clean", "--save-predictions"),
    },
    {
        "enabled": RUN_TEACHER_VALIDATION_PROMPT_SENSITIVITY,
        "label": "Teacher baseline: full validation, clean and noisy prompts",
        "command": python_script("scripts/run_teacher_baseline.py", "--split", "validation", "--prompt-mode", "both", "--run-tag", "validation_prompt_sensitivity", "--save-predictions"),
    },
    {
        "enabled": RUN_TEACHER_TEST_PROMPT_SENSITIVITY,
        "label": "Teacher baseline: full test, clean and noisy prompts",
        "command": python_script("scripts/run_teacher_baseline.py", "--split", "test", "--prompt-mode", "both", "--run-tag", "test_prompt_sensitivity", "--save-predictions"),
    },
    {
        "enabled": RUN_TEACHER_TRAIN_DISTILLATION_CLEAN,
        "label": "Teacher baseline: train split, clean prompts for distillation",
        "command": python_script("scripts/run_teacher_baseline.py", "--split", "train", "--prompt-mode", "clean", "--run-tag", "train_distillation_clean", "--save-predictions"),
    },
]

for spec in teacher_run_specs:
    run_project_command(spec["command"], enabled=spec["enabled"], label=spec["label"])

## Stage 3: LoRA Teacher Adaptation

This section makes LoRA an executable part of the study. It trains adapters, evaluates the LoRA-adapted teacher with saved prediction artifacts, and builds LoRA-specific distillation manifests that can feed a separate student run.


In [ ]:
from pathlib import Path
import subprocess
import sys
from datetime import datetime

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))

if "python_script" not in globals():
    def python_script(script_name, *args):
        return [sys.executable, str(REPO_ROOT / script_name), *map(str, args)]

def run_project_command_foreground(command, *, enabled=False, label=None):
    """Run a project command in the notebook foreground and stream output line-by-line."""
    display_label = label or " ".join(str(part) for part in command)
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {display_label}")
    print("Command:", " ".join(str(part) for part in command))
    if not enabled:
        print("Skipped. Set the matching RUN_* variable to True to execute this cell.")
        return None
    process = subprocess.Popen(
        command,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Completed: {display_label}")
    return return_code

RUN_LORA_TEACHER_PIPELINE = True
RUN_LORA_SKIP_TRAIN = True
RUN_LORA_SKIP_EVAL = False
RUN_LORA_SKIP_MANIFEST_BUILD = False

LORA_EPOCHS = 5
LORA_RANK = 8
LORA_ALPHA = 16.0
LORA_DROPOUT = 0.1
LORA_LEARNING_RATE = 1e-4
LORA_WEIGHT_DECAY = 1e-4
LORA_ACCUMULATION_STEPS = 4
LORA_DEVICE = "auto"
LORA_PRECISION = "fp32"
LORA_INPUT_SIZE = 256
LORA_PROGRESS_INTERVAL = 10
LORA_EVALUATION_PROMPT_MODE = "clean"
LORA_ALLOW_FALLBACK = False
LORA_TRAIN_MASK_DECODER = False

# Use these limits for a fast pilot. Keep them as None for the full study run.
LORA_TRAIN_SAMPLE_LIMIT = 256
LORA_VALIDATION_SAMPLE_LIMIT = 64
LORA_EVAL_SAMPLE_LIMIT = None

LORA_ADAPTER_OUTPUT = REPO_ROOT / "results" / "models" / "lora_teacher" / "rank8" / "adapters.pt"

lora_pipeline_args = [
    "--epochs", LORA_EPOCHS,
    "--rank", LORA_RANK,
    "--alpha", LORA_ALPHA,
    "--dropout", LORA_DROPOUT,
    "--learning-rate", LORA_LEARNING_RATE,
    "--weight-decay", LORA_WEIGHT_DECAY,
    "--accumulation-steps", LORA_ACCUMULATION_STEPS,
    "--device", LORA_DEVICE,
    "--precision", LORA_PRECISION,
    "--input-size", LORA_INPUT_SIZE,
    "--progress-interval", LORA_PROGRESS_INTERVAL,
    "--evaluation-prompt-mode", LORA_EVALUATION_PROMPT_MODE,
    "--adapter-output", LORA_ADAPTER_OUTPUT,
]

if RUN_LORA_SKIP_TRAIN:
    lora_pipeline_args.append("--skip-train")
if RUN_LORA_SKIP_EVAL:
    lora_pipeline_args.append("--skip-eval")
if RUN_LORA_SKIP_MANIFEST_BUILD:
    lora_pipeline_args.append("--skip-manifest-build")
if LORA_ALLOW_FALLBACK:
    lora_pipeline_args.append("--allow-fallback")
if LORA_TRAIN_MASK_DECODER:
    lora_pipeline_args.append("--train-mask-decoder")
if LORA_TRAIN_SAMPLE_LIMIT is not None:
    lora_pipeline_args.extend(["--train-sample-limit", LORA_TRAIN_SAMPLE_LIMIT])
if LORA_VALIDATION_SAMPLE_LIMIT is not None:
    lora_pipeline_args.extend(["--validation-sample-limit", LORA_VALIDATION_SAMPLE_LIMIT])
if LORA_EVAL_SAMPLE_LIMIT is not None:
    lora_pipeline_args.extend(["--eval-sample-limit", LORA_EVAL_SAMPLE_LIMIT])

lora_pipeline_command = python_script("scripts/run_lora_teacher_pipeline.py", *lora_pipeline_args)
run_project_command_foreground(
    lora_pipeline_command,
    enabled=RUN_LORA_TEACHER_PIPELINE,
    label="Run Stage 3 LoRA teacher pipeline in foreground",
)


### LoRA Teacher Outputs

Run this after the Stage 3 pipeline finishes to confirm the adapter, adapted-teacher metrics, and LoRA distillation manifests exist.


In [ ]:
from pathlib import Path
import pandas as pd

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()

lora_output_paths = {
    "adapter": REPO_ROOT / "results" / "models" / "lora_teacher" / "rank8" / "adapters.pt",
    "training_log": REPO_ROOT / "results" / "raw" / "lora_teacher_training_log.csv",
    "training_validation_summary": REPO_ROOT / "results" / "tables" / "lora_teacher_training_validation_summary.csv",
    "teacher_train_summary": REPO_ROOT / "results" / "tables" / "lora_teacher_kvasir_seg_train_clean_summary.csv",
    "teacher_validation_summary": REPO_ROOT / "results" / "tables" / "lora_teacher_kvasir_seg_validation_clean_summary.csv",
    "teacher_test_summary": REPO_ROOT / "results" / "tables" / "lora_teacher_kvasir_seg_test_clean_summary.csv",
    "train_manifest": REPO_ROOT / "results" / "raw" / "lora_teacher_distillation_manifest_kvasir_seg_train_clean.jsonl",
    "validation_manifest": REPO_ROOT / "results" / "raw" / "lora_teacher_distillation_manifest_kvasir_seg_validation_clean.jsonl",
    "test_manifest": REPO_ROOT / "results" / "raw" / "lora_teacher_distillation_manifest_kvasir_seg_test_clean.jsonl",
}

path_rows = []
for label, output_path in lora_output_paths.items():
    row = {"artifact": label, "exists": output_path.exists(), "path": output_path}
    if output_path.exists() and output_path.suffix == ".jsonl":
        row["records"] = sum(1 for line in output_path.read_text(encoding="utf-8").splitlines() if line.strip())
    path_rows.append(row)

display(pd.DataFrame(path_rows))

summary_paths = [
    lora_output_paths["training_validation_summary"],
    lora_output_paths["teacher_validation_summary"],
    lora_output_paths["teacher_test_summary"],
]
for summary_path in summary_paths:
    if summary_path.exists():
        print(summary_path.name)
        display(pd.read_csv(summary_path))


### Optional LoRA-Teacher Student Run

This keeps the original distilled student untouched and trains/evaluates a separate `student_distilled_lora_teacher` from the LoRA teacher manifests.


In [ ]:
from pathlib import Path
import subprocess
import sys
from datetime import datetime

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))

if "python_script" not in globals():
    def python_script(script_name, *args):
        return [sys.executable, str(REPO_ROOT / script_name), *map(str, args)]

if "run_project_command" not in globals():
    def run_project_command(command, *, enabled=False, label=None, stdout_mode="print"):
        display_label = label or " ".join(str(part) for part in command)
        timestamp = datetime.now().strftime("%H:%M:%S")
        print(f"[{timestamp}] {display_label}")
        print("Command:", " ".join(str(part) for part in command))
        if not enabled:
            print("Skipped. Set the matching RUN_* variable to True to execute this cell.")
            return None
        completed = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True)
        if completed.stdout:
            print(completed.stdout)
        if completed.stderr:
            print("STDERR:")
            print(completed.stderr)
        completed.check_returncode()
        return completed

RUN_LORA_STUDENT_DISTILLED_TRAINING = True
RUN_LORA_STUDENT_VALIDATION_EVAL = True
RUN_LORA_STUDENT_TEST_EVAL = True

LORA_STUDENT_MODEL_NAME = "student_distilled_lora_teacher"
LORA_STUDENT_MODEL_DIR = REPO_ROOT / "results" / "models" / LORA_STUDENT_MODEL_NAME
LORA_STUDENT_CHECKPOINT = LORA_STUDENT_MODEL_DIR / "best.pt"
LORA_STUDENT_TRAIN_MANIFEST = REPO_ROOT / "results" / "raw" / "lora_teacher_distillation_manifest_kvasir_seg_train_clean.jsonl"
LORA_STUDENT_VALIDATION_MANIFEST = REPO_ROOT / "results" / "raw" / "lora_teacher_distillation_manifest_kvasir_seg_validation_clean.jsonl"
LORA_STUDENT_TEST_MANIFEST = REPO_ROOT / "results" / "raw" / "lora_teacher_distillation_manifest_kvasir_seg_test_clean.jsonl"

lora_student_commands = [
    {
        "enabled": RUN_LORA_STUDENT_DISTILLED_TRAINING,
        "label": "Train U-Net student from LoRA teacher soft masks",
        "command": python_script(
            "scripts/train_student_distilled.py",
            "--train-manifest", LORA_STUDENT_TRAIN_MANIFEST,
            "--validation-manifest", LORA_STUDENT_VALIDATION_MANIFEST,
            "--epochs", 40,
            "--batch-size", 8,
            "--device", "auto",
            "--teacher-loss-weight", 0.1,
            "--model-name", LORA_STUDENT_MODEL_NAME,
            "--model-output", LORA_STUDENT_CHECKPOINT,
            "--training-log-output", REPO_ROOT / "results" / "raw" / f"{LORA_STUDENT_MODEL_NAME}_training_log.csv",
            "--raw-output", REPO_ROOT / "results" / "raw" / f"{LORA_STUDENT_MODEL_NAME}_validation.csv",
            "--summary-output", REPO_ROOT / "results" / "tables" / f"{LORA_STUDENT_MODEL_NAME}_summary.csv",
        ),
    },
    {
        "enabled": RUN_LORA_STUDENT_VALIDATION_EVAL,
        "label": "Evaluate LoRA-teacher student on validation",
        "command": python_script(
            "scripts/evaluate_student_model.py",
            "--split", "validation",
            "--manifest", LORA_STUDENT_VALIDATION_MANIFEST,
            "--model-name", LORA_STUDENT_MODEL_NAME,
            "--checkpoint", LORA_STUDENT_CHECKPOINT,
            "--run-tag", "validation_eval",
            "--save-predictions",
        ),
    },
    {
        "enabled": RUN_LORA_STUDENT_TEST_EVAL,
        "label": "Evaluate LoRA-teacher student on test",
        "command": python_script(
            "scripts/evaluate_student_model.py",
            "--split", "test",
            "--manifest", LORA_STUDENT_TEST_MANIFEST,
            "--model-name", LORA_STUDENT_MODEL_NAME,
            "--checkpoint", LORA_STUDENT_CHECKPOINT,
            "--run-tag", "test_eval",
            "--save-predictions",
        ),
    },
]

for spec in lora_student_commands:
    run_project_command(spec["command"], enabled=spec["enabled"], label=spec["label"])


## Teacher Baseline Analysis

This section loads validation and test teacher-baseline results, compares clean and noisy prompts, and prepares visual failure-case review. Tagged result files are preferred; existing untagged files are used as fallback until tagged runs are generated.


In [ ]:
import ast
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

TEACHER_ANALYSIS_RUNS = {"validation": "validation_prompt_sensitivity", "test": "test_prompt_sensitivity"}

def teacher_result_paths(split, run_tag=None):
    suffix = f"_{run_tag}" if run_tag else ""
    raw_path = REPO_ROOT / "results" / "raw" / f"teacher_baseline_kvasir_seg_{split}{suffix}.csv"
    summary_path = REPO_ROOT / "results" / "tables" / f"teacher_baseline_kvasir_seg_{split}{suffix}_summary.csv"
    return raw_path, summary_path

def load_teacher_result(split, run_tag):
    tagged_raw, tagged_summary = teacher_result_paths(split, run_tag)
    fallback_raw, fallback_summary = teacher_result_paths(split)
    raw_path = tagged_raw if tagged_raw.exists() else fallback_raw
    summary_path = tagged_summary if tagged_summary.exists() else fallback_summary
    source = "tagged" if tagged_raw.exists() and tagged_summary.exists() else "untagged fallback"
    if not raw_path.exists() or not summary_path.exists():
        print(f"Missing teacher result files for split={split}, run_tag={run_tag}.")
        print("Expected tagged raw:", tagged_raw)
        print("Expected tagged summary:", tagged_summary)
        return pd.DataFrame(), pd.DataFrame(), source
    raw = pd.read_csv(raw_path)
    summary = pd.read_csv(summary_path)
    raw["analysis_source"] = source
    summary["analysis_source"] = source
    print(f"Loaded {split}: {len(raw)} raw rows, {len(summary)} summary rows ({source}).")
    print("Raw:", raw_path)
    print("Summary:", summary_path)
    return raw, summary, source

teacher_raw_frames = []
teacher_summary_frames = []
for split, run_tag in TEACHER_ANALYSIS_RUNS.items():
    raw, summary, _ = load_teacher_result(split, run_tag)
    if not raw.empty:
        teacher_raw_frames.append(raw)
    if not summary.empty:
        teacher_summary_frames.append(summary)

teacher_rows = pd.concat(teacher_raw_frames, ignore_index=True) if teacher_raw_frames else pd.DataFrame()
teacher_summary = pd.concat(teacher_summary_frames, ignore_index=True) if teacher_summary_frames else pd.DataFrame()

In [ ]:
RUN_DISTILLATION_MANIFEST_BUILD = True

required_train_teacher_csv = REPO_ROOT / "results" / "raw" / "teacher_baseline_kvasir_seg_train_train_distillation_clean.csv"
required_train_prediction_dir = REPO_ROOT / "results" / "predictions" / "teacher_baseline" / "train" / "train_distillation_clean"
train_prediction_count = (
    len(list(required_train_prediction_dir.glob("*.npz")))
    if required_train_prediction_dir.exists()
    else 0
)

print("Required train teacher CSV:", required_train_teacher_csv)
print("Train teacher CSV exists:", required_train_teacher_csv.exists())
print("Train prediction artifacts:", train_prediction_count)

if RUN_DISTILLATION_MANIFEST_BUILD and (not required_train_teacher_csv.exists() or train_prediction_count == 0):
    print("Train distillation teacher artifacts are missing, so the manifest cannot be built yet.")
    print("First run the Teacher Baseline Runs cell with only this toggle set to True:")
    print("RUN_TEACHER_TRAIN_DISTILLATION_CLEAN = True")
    print("Expected after that run: 700 rows and 700 .npz prediction artifacts.")
elif RUN_DISTILLATION_MANIFEST_BUILD:
    distillation_manifest_command = python_script("scripts/build_teacher_distillation_manifest.py")
    run_project_command(
        distillation_manifest_command,
        enabled=True,
        label="Build teacher distillation JSONL manifests",
    )
else:
    print("Skipped. Set RUN_DISTILLATION_MANIFEST_BUILD = True after train teacher artifacts exist.")

In [ ]:
from deployable_medsam.distillation import load_distillation_preview, read_jsonl_manifest

DISTILLATION_MANIFEST_PATHS = {
    "train": REPO_ROOT / "results" / "raw" / "teacher_distillation_manifest_kvasir_seg_train_clean.jsonl",
    "validation": REPO_ROOT / "results" / "raw" / "teacher_distillation_manifest_kvasir_seg_validation_clean.jsonl",
    "test": REPO_ROOT / "results" / "raw" / "teacher_distillation_manifest_kvasir_seg_test_clean.jsonl",
}

distillation_records_by_split = {}
manifest_summary_rows = []
for split_name, manifest_path in DISTILLATION_MANIFEST_PATHS.items():
    if manifest_path.exists():
        records = read_jsonl_manifest(manifest_path)
        distillation_records_by_split[split_name] = records
        missing_artifacts = sum(
            1 for record in records
            if not (REPO_ROOT / record.teacher_prediction_path).exists()
        )
        manifest_summary_rows.append({
            "split": split_name,
            "records": len(records),
            "missing_artifacts": missing_artifacts,
            "manifest_path": manifest_path,
        })
    else:
        manifest_summary_rows.append({
            "split": split_name,
            "records": 0,
            "missing_artifacts": None,
            "manifest_path": manifest_path,
        })

manifest_summary = pd.DataFrame(manifest_summary_rows)
display(manifest_summary)
if manifest_summary["records"].sum() == 0:
    print("No distillation manifests found yet. Run train teacher artifacts first, then set RUN_DISTILLATION_MANIFEST_BUILD = True.")

In [ ]:
DISTILLATION_PREVIEW_SPLIT = "train"
DISTILLATION_PREVIEW_COUNT = 3

records = distillation_records_by_split.get(DISTILLATION_PREVIEW_SPLIT, [])
if not records:
    print(f"No {DISTILLATION_PREVIEW_SPLIT} distillation records available for preview.")
else:
    selected_records = records[:DISTILLATION_PREVIEW_COUNT]
    stats_rows = []
    fig, axes = plt.subplots(len(selected_records), 4, figsize=(16, 4 * len(selected_records)))
    if len(selected_records) == 1:
        axes = np.array([axes])

    for row_index, record in enumerate(selected_records):
        preview = load_distillation_preview(record, REPO_ROOT)
        image = preview["image"]
        ground_truth = np.asarray(preview["ground_truth_mask"], dtype=np.uint8)
        probabilities = preview["teacher_probabilities"]
        binary_mask = preview["teacher_binary_mask"]
        stats_rows.append({
            "sample_id": record.sample_id,
            "prob_min": float(probabilities.min()),
            "prob_mean": float(probabilities.mean()),
            "prob_max": float(probabilities.max()),
            "binary_foreground_pixels": int(binary_mask.sum()),
        })

        axes[row_index, 0].imshow(image)
        axes[row_index, 0].set_title(record.sample_id)
        axes[row_index, 1].imshow(ground_truth, cmap="gray", vmin=0, vmax=1)
        axes[row_index, 1].set_title("Ground truth")
        axes[row_index, 2].imshow(probabilities, cmap="viridis", vmin=0, vmax=1)
        axes[row_index, 2].set_title("Teacher probability")
        axes[row_index, 3].imshow(binary_mask, cmap="gray", vmin=0, vmax=1)
        axes[row_index, 3].set_title("Teacher binary")
        for axis in axes[row_index]:
            axis.axis("off")
    fig.tight_layout()
    plt.show()
    display(pd.DataFrame(stats_rows))

## Stage 5: Student Dataset and Lightweight U-Net Baseline

This section turns the Stage 4 distillation manifests into a trainable student dataset, previews teacher targets beside ground-truth masks, and runs the ground-truth-only lightweight U-Net baseline. Teacher probability masks are loaded here for inspection, but they are not used in the loss until the distillation stage.

In [ ]:
# Local helper fallback so this Stage 5 cell can run after a kernel restart.
from pathlib import Path
import subprocess
import sys
from datetime import datetime

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))

if "run_project_command" not in globals():
    def run_project_command(command, *, enabled=False, label=None, stdout_mode="print"):
        """Run a project command from the notebook and stream or summarize its output."""
        display_label = label or " ".join(str(part) for part in command)
        timestamp = datetime.now().strftime("%H:%M:%S")
        print(f"[{timestamp}] {display_label}")
        print("Command:", " ".join(str(part) for part in command))
        if not enabled:
            print("Skipped. Set the matching RUN_* variable to True to execute this cell.")
            return None

        completed = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True)
        if completed.stdout:
            if stdout_mode == "suppress":
                print(f"STDOUT suppressed ({len(completed.stdout)} characters).")
            else:
                print(completed.stdout)
        if completed.stderr:
            print("STDERR:")
            print(completed.stderr)
        completed.check_returncode()
        return completed

if "python_script" not in globals():
    def python_script(script_name, *args):
        return [sys.executable, str(REPO_ROOT / script_name), *map(str, args)]

RUN_STUDENT_DATASET_SMOKE = False
RUN_STUDENT_BASELINE_TRAINING = True

student_smoke_commands = [
    {"command": python_script("scripts/smoke_test_student_dataset.py"), "label": "Student dataset smoke test"},
    {"command": python_script("scripts/smoke_test_lightweight_unet.py"), "label": "Lightweight U-Net smoke test"},
    {"command": python_script("scripts/smoke_test_student_training_step.py"), "label": "Student training-step smoke test"},
]

for spec in student_smoke_commands:
    run_project_command(spec["command"], enabled=RUN_STUDENT_DATASET_SMOKE, label=spec["label"])

student_training_command = python_script(
    "scripts/train_student_baseline.py",
    "--epochs", 20,
    "--batch-size", 8,
    "--device", "auto",
)
run_project_command(
    student_training_command,
    enabled=RUN_STUDENT_BASELINE_TRAINING,
    label="Train lightweight U-Net student baseline",
)

In [ ]:
TRAIN_DISTILLATION_MANIFEST = REPO_ROOT / "results" / "raw" / "teacher_distillation_manifest_kvasir_seg_train_clean.jsonl"

try:
    import torch
    from deployable_medsam.student import build_student_dataloader
except ImportError as exc:
    print("Student dataset preview requires PyTorch.")
    print("Install experiment dependencies inside WSL with: python -m pip install -r requirements-experiments.txt")
    print("Import error:", exc)
else:
    if not TRAIN_DISTILLATION_MANIFEST.exists():
        print("Train distillation manifest is missing:", TRAIN_DISTILLATION_MANIFEST)
        print("Run Stage 4 after generating train teacher predictions before previewing the student dataset.")
    else:
        student_preview_loader = build_student_dataloader(
            TRAIN_DISTILLATION_MANIFEST,
            project_root=REPO_ROOT,
            batch_size=4,
            shuffle=False,
            num_workers=0,
        )
        student_preview_batch = next(iter(student_preview_loader))
        print("Batch image shape:", tuple(student_preview_batch["image"].shape))
        print("Batch ground-truth shape:", tuple(student_preview_batch["ground_truth_mask"].shape))
        print("Batch teacher probabilities shape:", tuple(student_preview_batch["teacher_probabilities"].shape))
        print("First sample ids:", list(student_preview_batch["sample_id"])[:4])

        preview_count = min(4, student_preview_batch["image"].shape[0])
        fig, axes = plt.subplots(preview_count, 4, figsize=(15, 3.8 * preview_count))
        if preview_count == 1:
            axes = np.array([axes])
        for row_index in range(preview_count):
            image = student_preview_batch["image"][row_index].permute(1, 2, 0).numpy()
            ground_truth = student_preview_batch["ground_truth_mask"][row_index, 0].numpy()
            teacher_probability = student_preview_batch["teacher_probabilities"][row_index, 0].numpy()
            teacher_binary = student_preview_batch["teacher_binary_mask"][row_index, 0].numpy()
            sample_id = student_preview_batch["sample_id"][row_index]

            axes[row_index, 0].imshow(image)
            axes[row_index, 0].set_title(f"{sample_id} image")
            axes[row_index, 1].imshow(ground_truth, cmap="gray", vmin=0, vmax=1)
            axes[row_index, 1].set_title("ground truth")
            axes[row_index, 2].imshow(teacher_probability, cmap="viridis", vmin=0, vmax=1)
            axes[row_index, 2].set_title("teacher probability")
            axes[row_index, 3].imshow(teacher_binary, cmap="gray", vmin=0, vmax=1)
            axes[row_index, 3].set_title("teacher binary")
            for col_index in range(4):
                axes[row_index, col_index].axis("off")
        plt.tight_layout()

In [ ]:
STUDENT_VALIDATION_RAW_PATH = REPO_ROOT / "results" / "raw" / "student_baseline_validation.csv"
STUDENT_SUMMARY_PATH = REPO_ROOT / "results" / "tables" / "student_baseline_summary.csv"

if STUDENT_SUMMARY_PATH.exists():
    student_summary_df = pd.read_csv(STUDENT_SUMMARY_PATH)
    display(student_summary_df)
else:
    print("Student baseline summary is not available yet:", STUDENT_SUMMARY_PATH)
    print("Set RUN_STUDENT_BASELINE_TRAINING = True to train the baseline from this notebook.")

if STUDENT_VALIDATION_RAW_PATH.exists():
    student_validation_df = pd.read_csv(STUDENT_VALIDATION_RAW_PATH)
    print("Student validation rows:", len(student_validation_df))
    display(student_validation_df.sort_values("dice").head(10))
else:
    print("Student validation raw metrics are not available yet:", STUDENT_VALIDATION_RAW_PATH)

In [ ]:
teacher_summary_candidates = [
    REPO_ROOT / "results" / "tables" / "teacher_baseline_kvasir_seg_validation_validation_prompt_sensitivity_summary.csv",
    REPO_ROOT / "results" / "tables" / "teacher_baseline_kvasir_seg_validation_validation_clean_summary.csv",
    REPO_ROOT / "results" / "tables" / "teacher_baseline_kvasir_seg_validation_summary.csv",
]

comparison_rows = []
for teacher_summary_path in teacher_summary_candidates:
    if teacher_summary_path.exists():
        teacher_summary_df = pd.read_csv(teacher_summary_path)
        teacher_clean_df = teacher_summary_df.copy()
        if "prompt_type" in teacher_clean_df.columns:
            teacher_clean_df = teacher_clean_df[teacher_clean_df["prompt_type"] == "clean"]
        if "jitter_pixels" in teacher_clean_df.columns:
            teacher_clean_df = teacher_clean_df[teacher_clean_df["jitter_pixels"] == 0]
        if not teacher_clean_df.empty:
            row = teacher_clean_df.iloc[0]
            comparison_rows.append({
                "model": "MedSAM teacher clean",
                "source": teacher_summary_path.name,
                "sample_count": int(row.get("sample_count", 0)),
                "mean_dice": float(row.get("mean_dice", np.nan)),
                "mean_iou": float(row.get("mean_iou", np.nan)),
                "mean_binary_ece_roi": float(row.get("mean_binary_ece_roi", np.nan)),
                "mean_binary_ece_boundary": float(row.get("mean_binary_ece_boundary", np.nan)),
                "mean_latency_ms": float(row.get("mean_latency_ms", np.nan)),
            })
            break

if STUDENT_SUMMARY_PATH.exists():
    student_summary_df = pd.read_csv(STUDENT_SUMMARY_PATH)
    if not student_summary_df.empty:
        row = student_summary_df.iloc[0]
        comparison_rows.append({
            "model": "Lightweight U-Net baseline",
            "source": STUDENT_SUMMARY_PATH.name,
            "sample_count": int(row.get("sample_count", 0)),
            "mean_dice": float(row.get("mean_dice", np.nan)),
            "mean_iou": float(row.get("mean_iou", np.nan)),
            "mean_binary_ece_roi": float(row.get("mean_binary_ece_roi", np.nan)),
            "mean_binary_ece_boundary": float(row.get("mean_binary_ece_boundary", np.nan)),
            "mean_latency_ms": float(row.get("mean_latency_ms", np.nan)),
        })

if comparison_rows:
    display(pd.DataFrame(comparison_rows))
else:
    print("No teacher/student summaries are available for comparison yet.")

## Stage 5.5 and Stage 6: Student Evaluation, Prediction Review, and Soft-Mask Distillation

This section evaluates saved student checkpoints on validation/test, saves prediction artifacts for visual review, trains the GT + teacher-soft-mask distilled U-Net, and builds the comparison table used for the next study narrative.

In [ ]:
# Local helper fallback so this cell can run after a kernel restart.
from pathlib import Path
import subprocess
import sys
from datetime import datetime

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))

if "run_project_command" not in globals():
    def run_project_command(command, *, enabled=False, label=None, stdout_mode="print"):
        display_label = label or " ".join(str(part) for part in command)
        timestamp = datetime.now().strftime("%H:%M:%S")
        print(f"[{timestamp}] {display_label}")
        print("Command:", " ".join(str(part) for part in command))
        if not enabled:
            print("Skipped. Set the matching RUN_* variable to True to execute this cell.")
            return None
        completed = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True)
        if completed.stdout:
            if stdout_mode == "suppress":
                print(f"STDOUT suppressed ({len(completed.stdout)} characters).")
            else:
                print(completed.stdout)
        if completed.stderr:
            print("STDERR:")
            print(completed.stderr)
        completed.check_returncode()
        return completed

if "python_script" not in globals():
    def python_script(script_name, *args):
        return [sys.executable, str(REPO_ROOT / script_name), *map(str, args)]


RUN_STUDENT_BASELINE_VALIDATION_EVAL = False
RUN_STUDENT_BASELINE_TEST_EVAL = False
RUN_STUDENT_DISTILLED_TRAINING = False
RUN_STUDENT_DISTILLATION_WEIGHT_SWEEP = False
RUN_STUDENT_DISTILLED_VALIDATION_EVAL = False
RUN_STUDENT_DISTILLED_TEST_EVAL = False
RUN_STUDENT_MODEL_COMPARISON = False

stage_5_5_and_6_commands = [
    {
        "enabled": RUN_STUDENT_BASELINE_VALIDATION_EVAL,
        "label": "Evaluate GT-only student baseline on validation and save predictions",
        "command": python_script(
            "scripts/evaluate_student_model.py",
            "--split", "validation",
            "--model-name", "student_baseline",
            "--checkpoint", REPO_ROOT / "results" / "models" / "student_baseline" / "best.pt",
            "--run-tag", "validation_eval",
            "--save-predictions",
        ),
    },
    {
        "enabled": RUN_STUDENT_BASELINE_TEST_EVAL,
        "label": "Evaluate GT-only student baseline on test and save predictions",
        "command": python_script(
            "scripts/evaluate_student_model.py",
            "--split", "test",
            "--model-name", "student_baseline",
            "--checkpoint", REPO_ROOT / "results" / "models" / "student_baseline" / "best.pt",
            "--run-tag", "test_eval",
            "--save-predictions",
        ),
    },
    {
        "enabled": RUN_STUDENT_DISTILLED_TRAINING,
        "label": "Train GT + teacher-soft-mask distilled U-Net",
        "command": python_script(
            "scripts/train_student_distilled.py",
            "--epochs", 20,
            "--batch-size", 8,
            "--device", "auto",
        ),
    },
    {
        "enabled": RUN_STUDENT_DISTILLATION_WEIGHT_SWEEP,
        "label": "Run teacher-loss-weight sweep for distilled U-Net",
        "command": python_script(
            "scripts/run_student_distillation_weight_sweep.py",
            "--epochs", 30,
            "--batch-size", 8,
            "--device", "auto",
            "--teacher-loss-weights", 0.0, 0.1, 0.25, 0.5, 0.75, 1.0,
            "--evaluate-best-test",
            "--save-best-predictions",
        ),
    },
    {
        "enabled": RUN_STUDENT_DISTILLED_VALIDATION_EVAL,
        "label": "Evaluate teacher-distilled student on validation and save predictions",
        "command": python_script(
            "scripts/evaluate_student_model.py",
            "--split", "validation",
            "--model-name", "student_distilled",
            "--checkpoint", REPO_ROOT / "results" / "models" / "student_distilled" / "best.pt",
            "--run-tag", "validation_eval",
            "--save-predictions",
        ),
    },
    {
        "enabled": RUN_STUDENT_DISTILLED_TEST_EVAL,
        "label": "Evaluate teacher-distilled student on test and save predictions",
        "command": python_script(
            "scripts/evaluate_student_model.py",
            "--split", "test",
            "--model-name", "student_distilled",
            "--checkpoint", REPO_ROOT / "results" / "models" / "student_distilled" / "best.pt",
            "--run-tag", "test_eval",
            "--save-predictions",
        ),
    },
    {
        "enabled": RUN_STUDENT_MODEL_COMPARISON,
        "label": "Build teacher/student comparison table",
        "command": python_script("scripts/compare_student_models.py"),
    },
]

for spec in stage_5_5_and_6_commands:
    run_project_command(spec["command"], enabled=spec["enabled"], label=spec["label"])

In [ ]:
from pathlib import Path
import sys

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd

STUDENT_DISTILLATION_SWEEP_PATH = REPO_ROOT / "results" / "tables" / "student_distillation_weight_sweep.csv"
STUDENT_DISTILLATION_SWEEP_BEST_PATH = REPO_ROOT / "results" / "tables" / "student_distillation_weight_sweep_best.csv"

if STUDENT_DISTILLATION_SWEEP_PATH.exists():
    sweep_df = pd.read_csv(STUDENT_DISTILLATION_SWEEP_PATH)
    display_columns = [
        "selection_rank", "weight_tag", "teacher_loss_weight", "epoch", "sample_count",
        "mean_dice", "mean_iou", "mean_recall", "mean_binary_ece_roi",
        "mean_binary_ece_boundary", "mean_latency_ms", "checkpoint_path",
    ]
    display(sweep_df[[column for column in display_columns if column in sweep_df.columns]])
else:
    print("Distillation weight sweep table is not available yet.")
    print("Set RUN_STUDENT_DISTILLATION_WEIGHT_SWEEP = False to run the one-command sweep.")

if STUDENT_DISTILLATION_SWEEP_BEST_PATH.exists():
    print("Selected best distillation weight")
    display(pd.read_csv(STUDENT_DISTILLATION_SWEEP_BEST_PATH))


In [ ]:
STUDENT_EVALUATION_SUMMARY_PATHS = {
    "baseline_validation": REPO_ROOT / "results" / "tables" / "student_student_baseline_validation_validation_eval_summary.csv",
    "baseline_test": REPO_ROOT / "results" / "tables" / "student_student_baseline_test_test_eval_summary.csv",
    "distilled_validation": REPO_ROOT / "results" / "tables" / "student_student_distilled_validation_validation_eval_summary.csv",
    "distilled_test": REPO_ROOT / "results" / "tables" / "student_student_distilled_test_test_eval_summary.csv",
    "distilled_training": REPO_ROOT / "results" / "tables" / "student_distilled_summary.csv",
    "distillation_sweep_best": REPO_ROOT / "results" / "tables" / "student_distillation_weight_sweep_best.csv",
    "distillation_sweep_best_test": REPO_ROOT / "results" / "tables" / "student_distillation_weight_sweep_best_test_summary.csv",
    "comparison": REPO_ROOT / "results" / "tables" / "student_model_comparison.csv",
}

available_student_summaries = []
for label, summary_path in STUDENT_EVALUATION_SUMMARY_PATHS.items():
    if summary_path.exists():
        df = pd.read_csv(summary_path)
        if not df.empty:
            row = df.iloc[0].to_dict()
            row["label"] = label
            row["source"] = summary_path.name
            available_student_summaries.append(row)

if available_student_summaries:
    summary_df = pd.DataFrame(available_student_summaries)
    display_columns = [
        "label", "source", "split", "model_id", "sample_count", "mean_dice", "mean_iou",
        "mean_binary_ece_roi", "mean_binary_ece_boundary", "mean_latency_ms", "trainable_parameters",
    ]
    display(summary_df[[column for column in display_columns if column in summary_df.columns]])
else:
    print("No student evaluation summaries are available yet. Run one of the Stage 5.5/6 evaluation toggles above.")

comparison_path = STUDENT_EVALUATION_SUMMARY_PATHS["comparison"]
if comparison_path.exists():
    display(pd.read_csv(comparison_path))
else:
    print("Comparison table is not available yet. Set RUN_STUDENT_MODEL_COMPARISON = True after evaluation/training outputs exist.")

In [ ]:
from pathlib import Path
import sys

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from deployable_medsam.data import load_binary_mask, load_rgb_image
from deployable_medsam.distillation import read_jsonl_manifest
from deployable_medsam.distillation.artifacts import resolve_project_path

STUDENT_REVIEW_MODEL_NAME = "student_distilled"
STUDENT_REVIEW_SPLIT = "validation"
STUDENT_REVIEW_RUN_TAG = "validation_eval"
STUDENT_REVIEW_COUNT = 5

review_candidates = [
    (STUDENT_REVIEW_MODEL_NAME, STUDENT_REVIEW_SPLIT, STUDENT_REVIEW_RUN_TAG),
    ("student_distilled", "validation", "validation_eval"),
    ("student_baseline", "validation", "validation_eval"),
    ("student_distilled", "test", "test_eval"),
    ("student_baseline", "test", "test_eval"),
]
seen_candidates = set()
selected_review = None
for model_name, split_name, run_tag in review_candidates:
    key = (model_name, split_name, run_tag)
    if key in seen_candidates:
        continue
    seen_candidates.add(key)
    raw_path = REPO_ROOT / "results" / "raw" / f"student_{model_name}_{split_name}_{run_tag}.csv"
    manifest_path = REPO_ROOT / "results" / "raw" / f"teacher_distillation_manifest_kvasir_seg_{split_name}_clean.jsonl"
    if raw_path.exists() and manifest_path.exists():
        preview_df = pd.read_csv(raw_path)
        if "prediction_path" in preview_df.columns:
            selected_review = (model_name, split_name, run_tag, raw_path, manifest_path, preview_df)
            break

if selected_review is None:
    expected_path = REPO_ROOT / "results" / "raw" / f"student_{STUDENT_REVIEW_MODEL_NAME}_{STUDENT_REVIEW_SPLIT}_{STUDENT_REVIEW_RUN_TAG}.csv"
    print("No student prediction CSV with prediction_path is available for visual review.")
    print("Expected first choice:", expected_path)
    print("Run the matching Stage 5.5 evaluation toggle with --save-predictions enabled.")
else:
    selected_model_name, selected_split, selected_run_tag, student_raw_path, student_manifest_path, student_rows = selected_review
    if (selected_model_name, selected_split, selected_run_tag) != (STUDENT_REVIEW_MODEL_NAME, STUDENT_REVIEW_SPLIT, STUDENT_REVIEW_RUN_TAG):
        print(
            "Requested review artifacts were not available; using available artifacts:",
            selected_model_name,
            selected_split,
            selected_run_tag,
        )
    print("Visual review CSV:", student_raw_path)

    manifest_records = {record.sample_id: record for record in read_jsonl_manifest(student_manifest_path)}
    print("Worst Dice cases")
    display(student_rows.sort_values("dice").head(STUDENT_REVIEW_COUNT)[[
        "sample_id", "dice", "iou", "precision", "recall", "binary_ece_roi", "binary_ece_boundary", "prediction_path"
    ]])
    print("Highest ROI ECE cases")
    display(student_rows.sort_values("binary_ece_roi", ascending=False).head(STUDENT_REVIEW_COUNT)[[
        "sample_id", "dice", "iou", "binary_ece_roi", "binary_ece_boundary", "prediction_path"
    ]])
    print("Highest boundary ECE cases")
    display(student_rows.sort_values("binary_ece_boundary", ascending=False).head(STUDENT_REVIEW_COUNT)[[
        "sample_id", "dice", "iou", "binary_ece_roi", "binary_ece_boundary", "prediction_path"
    ]])

    selected = student_rows.sort_values("dice").head(STUDENT_REVIEW_COUNT)
    fig, axes = plt.subplots(len(selected), 5, figsize=(18, 3.6 * len(selected)))
    if len(selected) == 1:
        axes = np.array([axes])
    for row_index, (_, metric_row) in enumerate(selected.iterrows()):
        record = manifest_records.get(metric_row["sample_id"])
        if record is None:
            for col_index in range(5):
                axes[row_index, col_index].axis("off")
            axes[row_index, 0].set_title(f"Missing manifest record: {metric_row['sample_id']}")
            continue
        image = load_rgb_image(resolve_project_path(record.image_path, REPO_ROOT), size=(record.input_size, record.input_size))
        ground_truth = np.asarray(load_binary_mask(resolve_project_path(record.mask_path, REPO_ROOT), size=(record.input_size, record.input_size)), dtype=np.uint8)
        prediction_path = resolve_project_path(metric_row["prediction_path"], REPO_ROOT)
        with np.load(prediction_path) as artifact:
            probabilities = artifact["probabilities"]
            binary_mask = artifact["binary_mask"].astype(np.uint8)

        true_positive = (binary_mask == 1) & (ground_truth == 1)
        false_positive = (binary_mask == 1) & (ground_truth == 0)
        false_negative = (binary_mask == 0) & (ground_truth == 1)
        error_overlay = np.zeros((*ground_truth.shape, 3), dtype=np.float32)
        error_overlay[true_positive] = [0.0, 0.8, 0.2]
        error_overlay[false_positive] = [1.0, 0.1, 0.1]
        error_overlay[false_negative] = [0.1, 0.3, 1.0]

        axes[row_index, 0].imshow(image)
        axes[row_index, 0].set_title(f"{metric_row['sample_id']} image")
        axes[row_index, 1].imshow(ground_truth, cmap="gray", vmin=0, vmax=1)
        axes[row_index, 1].set_title("ground truth")
        axes[row_index, 2].imshow(probabilities, cmap="viridis", vmin=0, vmax=1)
        axes[row_index, 2].set_title("prediction probability")
        axes[row_index, 3].imshow(binary_mask, cmap="gray", vmin=0, vmax=1)
        axes[row_index, 3].set_title("prediction binary")
        axes[row_index, 4].imshow(image)
        axes[row_index, 4].imshow(error_overlay, alpha=0.55)
        axes[row_index, 4].set_title(f"error overlay Dice={metric_row['dice']:.3f}")
        for col_index in range(5):
            axes[row_index, col_index].axis("off")
    plt.tight_layout()


## Stage 7: INT8 Quantization and Deployment Evaluation

This section exports the selected best distilled student to FP32 ONNX, applies ONNX Runtime static INT8 QDQ quantization using train-split calibration samples, evaluates INT8 validation/test behavior, and refreshes the final comparison table. All execution toggles default to `False`.

In [ ]:
# Local helper fallback so this Stage 7 cell can run after a kernel restart.
from pathlib import Path
import subprocess
import sys
from datetime import datetime

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))

if "python_script" not in globals():
    def python_script(script_name, *args):
        return [sys.executable, str(REPO_ROOT / script_name), *map(str, args)]

def run_project_command_foreground(command, *, enabled=False, label=None):
    """Run a project command in the notebook foreground and stream output line-by-line."""
    display_label = label or " ".join(str(part) for part in command)
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {display_label}")
    print("Command:", " ".join(str(part) for part in command))
    if not enabled:
        print("Skipped. Set the matching RUN_* variable to True to execute this cell.")
        return None
    process = subprocess.Popen(
        command,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Completed: {display_label}")
    return return_code

RUN_STUDENT_ONNX_EXPORT = False
RUN_STUDENT_ONNX_QUANTIZATION = False
RUN_STUDENT_INT8_VALIDATION_EVAL = False
RUN_STUDENT_INT8_TEST_EVAL = False
RUN_STUDENT_QUANTIZATION_PIPELINE = True
RUN_STUDENT_MODEL_COMPARISON = False

SELECTED_DISTILLED_MODEL_NAME = "student_distilled_lora_teacher"
SELECTED_DISTILLED_FP32_ONNX_MODEL_NAME = f"{SELECTED_DISTILLED_MODEL_NAME}_fp32_onnx"
SELECTED_DISTILLED_INT8_MODEL_NAME = f"{SELECTED_DISTILLED_MODEL_NAME}_int8"

SELECTED_DISTILLED_CHECKPOINT = REPO_ROOT / "results" / "models" / SELECTED_DISTILLED_MODEL_NAME / "best.pt"
SELECTED_DISTILLED_MODEL_DIR = REPO_ROOT / "results" / "models" / SELECTED_DISTILLED_MODEL_NAME
SELECTED_DISTILLED_FP32_ONNX = SELECTED_DISTILLED_MODEL_DIR / "fp32.onnx"
SELECTED_DISTILLED_INT8_ONNX = SELECTED_DISTILLED_MODEL_DIR / "int8_qdq.onnx"
SELECTED_DISTILLED_TRAIN_MANIFEST = REPO_ROOT / "results" / "raw" / "lora_teacher_distillation_manifest_kvasir_seg_train_clean.jsonl"
SELECTED_DISTILLED_VALIDATION_MANIFEST = REPO_ROOT / "results" / "raw" / "lora_teacher_distillation_manifest_kvasir_seg_validation_clean.jsonl"
SELECTED_DISTILLED_TEST_MANIFEST = REPO_ROOT / "results" / "raw" / "lora_teacher_distillation_manifest_kvasir_seg_test_clean.jsonl"
SELECTED_DISTILLED_COMPARISON = REPO_ROOT / "results" / "tables" / "student_lora_model_comparison.csv"

stage_7_commands = [
    {
        "enabled": RUN_STUDENT_ONNX_EXPORT,
        "label": "Export selected distilled student to FP32 ONNX",
        "command": python_script(
            "scripts/export_student_onnx.py",
            "--checkpoint", SELECTED_DISTILLED_CHECKPOINT,
            "--output", SELECTED_DISTILLED_FP32_ONNX,
        ),
    },
    {
        "enabled": RUN_STUDENT_ONNX_QUANTIZATION,
        "label": "Quantize selected distilled student ONNX to static INT8 QDQ",
        "command": python_script(
            "scripts/quantize_student_onnx.py",
            "--fp32-onnx", SELECTED_DISTILLED_FP32_ONNX,
            "--int8-onnx", SELECTED_DISTILLED_INT8_ONNX,
            "--calibration-manifest", SELECTED_DISTILLED_TRAIN_MANIFEST,
            "--calibration-samples", 128,
            "--calibration-method", "minmax",
        ),
    },
    {
        "enabled": RUN_STUDENT_INT8_VALIDATION_EVAL,
        "label": "Evaluate INT8 distilled student on validation and save predictions",
        "command": python_script(
            "scripts/evaluate_onnx_student_model.py",
            "--split", "validation",
            "--manifest", SELECTED_DISTILLED_VALIDATION_MANIFEST,
            "--model-name", SELECTED_DISTILLED_INT8_MODEL_NAME,
            "--onnx-model", SELECTED_DISTILLED_INT8_ONNX,
            "--run-tag", "quantized_eval",
            "--batch-size", 1,
            "--save-predictions",
        ),
    },
    {
        "enabled": RUN_STUDENT_INT8_TEST_EVAL,
        "label": "Evaluate INT8 distilled student on test and save predictions",
        "command": python_script(
            "scripts/evaluate_onnx_student_model.py",
            "--split", "test",
            "--manifest", SELECTED_DISTILLED_TEST_MANIFEST,
            "--model-name", SELECTED_DISTILLED_INT8_MODEL_NAME,
            "--onnx-model", SELECTED_DISTILLED_INT8_ONNX,
            "--run-tag", "quantized_eval",
            "--batch-size", 1,
            "--save-predictions",
        ),
    },
    {
        "enabled": RUN_STUDENT_QUANTIZATION_PIPELINE,
        "label": "Run LoRA-student ONNX export, INT8 quantization, and validation/test evaluation",
        "command": python_script(
            "scripts/run_student_quantization_pipeline.py",
            "--checkpoint", SELECTED_DISTILLED_CHECKPOINT,
            "--model-dir", SELECTED_DISTILLED_MODEL_DIR,
            "--model-name", SELECTED_DISTILLED_INT8_MODEL_NAME,
            "--train-manifest", SELECTED_DISTILLED_TRAIN_MANIFEST,
            "--validation-manifest", SELECTED_DISTILLED_VALIDATION_MANIFEST,
            "--test-manifest", SELECTED_DISTILLED_TEST_MANIFEST,
            "--calibration-samples", 128,
            "--calibration-method", "minmax",
            "--batch-size", 1,
            "--evaluate-validation",
            "--evaluate-test",
            "--save-predictions",
            "--skip-comparison",
        ),
    },
    {
        "enabled": RUN_STUDENT_MODEL_COMPARISON,
        "label": "Refresh LoRA teacher/student/INT8 comparison table",
        "command": python_script(
            "scripts/compare_student_models.py",
            "--output", SELECTED_DISTILLED_COMPARISON,
            "--fp32-distilled-model-path", SELECTED_DISTILLED_FP32_ONNX,
            "--int8-model-path", SELECTED_DISTILLED_INT8_ONNX,
            "--teacher-summary-candidates",
            REPO_ROOT / "results" / "tables" / "lora_teacher_kvasir_seg_validation_clean_summary.csv",
            "--distilled-summary-candidates",
            REPO_ROOT / "results" / "tables" / f"{SELECTED_DISTILLED_FP32_ONNX_MODEL_NAME}_validation_onnx_cpu_eval_summary.csv",
            REPO_ROOT / "results" / "tables" / f"{SELECTED_DISTILLED_MODEL_NAME}_summary.csv",
            "--quantized-summary-candidates",
            REPO_ROOT / "results" / "tables" / f"{SELECTED_DISTILLED_INT8_MODEL_NAME}_validation_quantized_eval_summary.csv",
        ),
    },
]

for item in stage_7_commands:
    run_project_command_foreground(item["command"], enabled=item["enabled"], label=item["label"])


In [ ]:
from pathlib import Path
import sys

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd

LORA_STUDENT_MODEL_NAME = "student_distilled_lora_teacher"
LORA_STUDENT_INT8_MODEL_NAME = f"{LORA_STUDENT_MODEL_NAME}_int8"

QUANTIZED_SUMMARY_PATHS = {
    "distilled_fp32_validation": REPO_ROOT / "results" / "tables" / f"{LORA_STUDENT_MODEL_NAME}_summary.csv",
    "distilled_fp32_test": REPO_ROOT / "results" / "tables" / f"student_{LORA_STUDENT_MODEL_NAME}_test_test_eval_summary.csv",
    "distilled_int8_validation": REPO_ROOT / "results" / "tables" / f"{LORA_STUDENT_INT8_MODEL_NAME}_validation_quantized_eval_summary.csv",
    "distilled_int8_test": REPO_ROOT / "results" / "tables" / f"{LORA_STUDENT_INT8_MODEL_NAME}_test_quantized_eval_summary.csv",
    "comparison": REPO_ROOT / "results" / "tables" / "student_lora_model_comparison.csv",
}

quantized_summary_rows = []
for label, path in QUANTIZED_SUMMARY_PATHS.items():
    if label == "comparison":
        continue
    if path.exists():
        df = pd.read_csv(path)
        if not df.empty:
            row = df.iloc[0].to_dict()
            row["label"] = label
            row["source"] = path.name
            quantized_summary_rows.append(row)

if quantized_summary_rows:
    quantized_summary_df = pd.DataFrame(quantized_summary_rows)
    display_columns = [
        "label", "source", "split", "model_id", "sample_count", "mean_dice", "mean_iou",
        "mean_precision", "mean_recall", "mean_binary_ece_roi", "mean_binary_ece_boundary",
        "mean_latency_ms", "onnx_model_size_mb", "fp32_model_size_mb", "int8_model_size_mb",
        "compression_ratio_vs_fp32", "calibration_method",
    ]
    display(quantized_summary_df[[column for column in display_columns if column in quantized_summary_df.columns]])

    if {"distilled_fp32_validation", "distilled_int8_validation"}.issubset(set(quantized_summary_df["label"])):
        fp32_row = quantized_summary_df[quantized_summary_df["label"] == "distilled_fp32_validation"].iloc[0]
        int8_row = quantized_summary_df[quantized_summary_df["label"] == "distilled_int8_validation"].iloc[0]
        print("Validation FP32 vs INT8")
        print("Dice drop:", round(float(fp32_row["mean_dice"]) - float(int8_row["mean_dice"]), 6))
        if "mean_latency_ms" in fp32_row and "mean_latency_ms" in int8_row and float(int8_row["mean_latency_ms"]) > 0:
            print("Speedup vs FP32 distilled:", round(float(fp32_row["mean_latency_ms"]) / float(int8_row["mean_latency_ms"]), 6))
else:
    print("No Stage 7 quantized summaries are available yet.")
    print("Run RUN_STUDENT_QUANTIZATION_PIPELINE = True, or run export, quantization, and INT8 evaluation toggles in order.")

comparison_path = QUANTIZED_SUMMARY_PATHS["comparison"]
if comparison_path.exists():
    display(pd.read_csv(comparison_path))
else:
    print("Comparison table is not available yet. Set RUN_STUDENT_MODEL_COMPARISON = True after Stage 7 evaluation.")


In [ ]:
from pathlib import Path
import sys

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from deployable_medsam.data import load_binary_mask, load_rgb_image
from deployable_medsam.distillation import read_jsonl_manifest
from deployable_medsam.distillation.artifacts import resolve_project_path

INT8_REVIEW_SPLIT = "test"
INT8_REVIEW_RUN_TAG = "quantized_eval"
INT8_REVIEW_COUNT = 5
INT8_REVIEW_MODEL_NAME = "student_distilled_lora_teacher_int8"

int8_raw_path = REPO_ROOT / "results" / "raw" / f"{INT8_REVIEW_MODEL_NAME}_{INT8_REVIEW_SPLIT}_{INT8_REVIEW_RUN_TAG}.csv"
int8_manifest_path = REPO_ROOT / "results" / "raw" / f"lora_teacher_distillation_manifest_kvasir_seg_{INT8_REVIEW_SPLIT}_clean.jsonl"

if not int8_raw_path.exists() or not int8_manifest_path.exists():
    print("INT8 visual review artifacts are not available yet.")
    print("Expected raw CSV:", int8_raw_path)
    print("Run RUN_STUDENT_QUANTIZATION_PIPELINE = True or the matching INT8 evaluation toggle with --save-predictions.")
elif "prediction_path" not in pd.read_csv(int8_raw_path, nrows=1).columns:
    print("INT8 raw CSV exists, but it does not include prediction_path.")
    print("Rerun INT8 evaluation with --save-predictions enabled.")
else:
    int8_rows = pd.read_csv(int8_raw_path)
    records = {record.sample_id: record for record in read_jsonl_manifest(int8_manifest_path)}
    review_cases = pd.concat([
        int8_rows.sort_values("dice", ascending=True).head(INT8_REVIEW_COUNT),
        int8_rows.sort_values("binary_ece_roi", ascending=False).head(INT8_REVIEW_COUNT),
        int8_rows.sort_values("binary_ece_boundary", ascending=False).head(INT8_REVIEW_COUNT),
    ]).drop_duplicates("sample_id").head(INT8_REVIEW_COUNT)

    display_columns = ["sample_id", "dice", "iou", "binary_ece_roi", "binary_ece_boundary", "latency_ms", "prediction_path"]
    display(review_cases[[column for column in display_columns if column in review_cases.columns]])

    for _, row in review_cases.iterrows():
        sample_id = str(row["sample_id"])
        record = records[sample_id]
        image = np.asarray(load_rgb_image(resolve_project_path(record.image_path, REPO_ROOT), size=(record.input_size, record.input_size)))
        target = np.asarray(load_binary_mask(resolve_project_path(record.mask_path, REPO_ROOT), size=(record.input_size, record.input_size)), dtype=bool)
        prediction_path = resolve_project_path(row["prediction_path"], REPO_ROOT)
        with np.load(prediction_path) as artifact:
            probability = artifact["probabilities"].astype(np.float32)
            predicted = artifact["binary_mask"].astype(bool)

        overlay = np.zeros((*target.shape, 3), dtype=np.float32)
        overlay[target & predicted] = [0.0, 0.8, 0.2]
        overlay[~target & predicted] = [1.0, 0.25, 0.0]
        overlay[target & ~predicted] = [0.1, 0.35, 1.0]

        fig, axes = plt.subplots(1, 5, figsize=(15, 3))
        fig.suptitle(f"{sample_id} | Dice={float(row['dice']):.3f} | ROI ECE={float(row['binary_ece_roi']):.3f}")
        axes[0].imshow(image)
        axes[0].set_title("Image")
        axes[1].imshow(target, cmap="gray")
        axes[1].set_title("Ground truth")
        axes[2].imshow(probability, cmap="viridis", vmin=0, vmax=1)
        axes[2].set_title("INT8 probability")
        axes[3].imshow(predicted, cmap="gray")
        axes[3].set_title("INT8 mask")
        axes[4].imshow(image)
        axes[4].imshow(overlay, alpha=0.55)
        axes[4].set_title("TP / FP / FN")
        for axis in axes:
            axis.axis("off")
        plt.tight_layout()
        plt.show()


## Stage 8: Quantization Calibration and Threshold Ablation

This section makes the INT8 result scientifically defensible: it evaluates FP32 ONNX on the same CPU runtime, re-evaluates INT8 with a validation-selected threshold, and optionally runs a calibration ablation over sample count, calibration method, and ONNX preprocessing.

In [ ]:
from pathlib import Path
import subprocess
import sys
from datetime import datetime

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))

def run_project_command_foreground(command, *, enabled=False, label=None):
    """Run a project command in the notebook foreground and stream output line-by-line."""
    display_label = label or " ".join(str(part) for part in command)
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {display_label}")
    print("Command:", " ".join(str(part) for part in command))
    if not enabled:
        print("Skipped. Set the matching RUN_* variable to True to execute this cell.")
        return None
    process = subprocess.Popen(
        command,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Completed: {display_label}")
    return return_code

if "python_script" not in globals():
    def python_script(script_name, *args):
        return [sys.executable, str(REPO_ROOT / script_name), *map(str, args)]

RUN_FP32_ONNX_VALIDATION_EVAL = False
RUN_FP32_ONNX_TEST_EVAL = False
RUN_INT8_THRESHOLD_SWEEP = False
RUN_INT8_THRESHOLD_085_VALIDATION_EVAL = False
RUN_INT8_THRESHOLD_085_TEST_EVAL = False

RUN_STAGE8_MINMAX_ABLATION = False
RUN_STAGE8_PERCENTILE_SMALL_ABLATION = False
RUN_STAGE8_PERCENTILE_128_RAW = False
RUN_STAGE8_PERCENTILE_128_PREPROCESSED = False
RUN_STAGE8_PERCENTILE_256_RAW = False
RUN_STAGE8_PERCENTILE_256_PREPROCESSED = False
RUN_STAGE8_FINALIZE_BEST = True
RUN_STUDENT_MODEL_COMPARISON = False

STAGE8_STUDENT_MODEL_NAME = "student_distilled_lora_teacher"
STAGE8_FP32_ONNX_MODEL_NAME = f"{STAGE8_STUDENT_MODEL_NAME}_fp32_onnx"
STAGE8_INT8_MODEL_NAME = f"{STAGE8_STUDENT_MODEL_NAME}_int8"
STAGE8_INT8_T085_MODEL_NAME = f"{STAGE8_INT8_MODEL_NAME}_t085"
STAGE8_MODEL_DIR = REPO_ROOT / "results" / "models" / STAGE8_STUDENT_MODEL_NAME
STAGE8_CHECKPOINT = STAGE8_MODEL_DIR / "best.pt"
FP32_ONNX_PATH = STAGE8_MODEL_DIR / "fp32.onnx"
INT8_ONNX_PATH = STAGE8_MODEL_DIR / "int8_qdq.onnx"
STAGE8_TRAIN_MANIFEST = REPO_ROOT / "results" / "raw" / "lora_teacher_distillation_manifest_kvasir_seg_train_clean.jsonl"
STAGE8_VALIDATION_MANIFEST = REPO_ROOT / "results" / "raw" / "lora_teacher_distillation_manifest_kvasir_seg_validation_clean.jsonl"
STAGE8_TEST_MANIFEST = REPO_ROOT / "results" / "raw" / "lora_teacher_distillation_manifest_kvasir_seg_test_clean.jsonl"
STAGE8_SELECTED_THRESHOLD = 0.85

STAGE8_COMMON_ABLATION_ARGS = [
    "--checkpoint", STAGE8_CHECKPOINT,
    "--fp32-onnx", FP32_ONNX_PATH,
    "--model-dir", STAGE8_MODEL_DIR,
    "--ablation-root", STAGE8_MODEL_DIR / "stage8_ablation",
    "--train-manifest", STAGE8_TRAIN_MANIFEST,
    "--validation-manifest", STAGE8_VALIDATION_MANIFEST,
    "--test-manifest", STAGE8_TEST_MANIFEST,
    "--ablation-output", REPO_ROOT / "results" / "tables" / "stage8_lora_quantization_ablation_validation.csv",
    "--best-output", REPO_ROOT / "results" / "tables" / "stage8_lora_quantization_ablation_best.csv",
    "--thresholds", "0.5:0.95:0.025",
    "--batch-size", 1,
    "--save-predictions",
    "--resume",
    "--skip-existing",
    "--no-evaluate-fp32",
    "--no-evaluate-test-best",
]

def stage8_ablation_command(*args):
    return python_script("scripts/run_stage8_quantization_ablation.py", *STAGE8_COMMON_ABLATION_ARGS, *args)

stage_8_commands = [
    {
        "enabled": RUN_FP32_ONNX_VALIDATION_EVAL,
        "label": "Evaluate FP32 ONNX student on validation CPU runtime",
        "command": python_script(
            "scripts/evaluate_onnx_student_model.py",
            "--split", "validation",
            "--manifest", STAGE8_VALIDATION_MANIFEST,
            "--model-name", STAGE8_FP32_ONNX_MODEL_NAME,
            "--onnx-model", FP32_ONNX_PATH,
            "--run-tag", "onnx_cpu_eval",
            "--batch-size", 1,
            "--save-predictions",
        ),
    },
    {
        "enabled": RUN_FP32_ONNX_TEST_EVAL,
        "label": "Evaluate FP32 ONNX student on test CPU runtime",
        "command": python_script(
            "scripts/evaluate_onnx_student_model.py",
            "--split", "test",
            "--manifest", STAGE8_TEST_MANIFEST,
            "--model-name", STAGE8_FP32_ONNX_MODEL_NAME,
            "--onnx-model", FP32_ONNX_PATH,
            "--run-tag", "onnx_cpu_eval",
            "--batch-size", 1,
            "--save-predictions",
        ),
    },
    {
        "enabled": RUN_INT8_THRESHOLD_SWEEP,
        "label": "Run fast INT8 validation threshold sweep",
        "command": python_script(
            "scripts/run_onnx_threshold_sweep.py",
            "--prediction-csv", REPO_ROOT / "results" / "raw" / f"{STAGE8_INT8_MODEL_NAME}_validation_quantized_eval.csv",
            "--manifest", STAGE8_VALIDATION_MANIFEST,
            "--output", REPO_ROOT / "results" / "tables" / f"{STAGE8_INT8_MODEL_NAME}_validation_threshold_sweep.csv",
            "--thresholds", "0.5:0.95:0.025",
            "--label", "lora_stage7_int8_minmax_s128",
        ),
    },
    {
        "enabled": RUN_INT8_THRESHOLD_085_VALIDATION_EVAL,
        "label": "Evaluate INT8 distilled student on validation at threshold 0.85",
        "command": python_script(
            "scripts/evaluate_onnx_student_model.py",
            "--split", "validation",
            "--manifest", STAGE8_VALIDATION_MANIFEST,
            "--model-name", STAGE8_INT8_T085_MODEL_NAME,
            "--onnx-model", INT8_ONNX_PATH,
            "--run-tag", "quantized_eval_t085",
            "--threshold", STAGE8_SELECTED_THRESHOLD,
            "--batch-size", 1,
            "--save-predictions",
        ),
    },
    {
        "enabled": RUN_INT8_THRESHOLD_085_TEST_EVAL,
        "label": "Evaluate INT8 distilled student on test at threshold 0.85",
        "command": python_script(
            "scripts/evaluate_onnx_student_model.py",
            "--split", "test",
            "--manifest", STAGE8_TEST_MANIFEST,
            "--model-name", STAGE8_INT8_T085_MODEL_NAME,
            "--onnx-model", INT8_ONNX_PATH,
            "--run-tag", "quantized_eval_t085",
            "--threshold", STAGE8_SELECTED_THRESHOLD,
            "--batch-size", 1,
            "--save-predictions",
        ),
    },
    {
        "enabled": RUN_STAGE8_MINMAX_ABLATION,
        "label": "Run Stage 8 MinMax ablation chunk in foreground",
        "command": stage8_ablation_command(
            "--calibration-samples", 32, 64, 128, 256,
            "--calibration-methods", "minmax",
            "--preprocessing-modes", "none", "preprocess",
        ),
    },
    {
        "enabled": RUN_STAGE8_PERCENTILE_SMALL_ABLATION,
        "label": "Run Stage 8 small Percentile ablation chunk in foreground",
        "command": stage8_ablation_command(
            "--calibration-samples", 32, 64,
            "--calibration-methods", "percentile",
            "--preprocessing-modes", "none", "preprocess",
        ),
    },
    {
        "enabled": RUN_STAGE8_PERCENTILE_128_RAW,
        "label": "Run optional Stage 8 Percentile 128 raw candidate in foreground",
        "command": stage8_ablation_command("--candidate-tags", "s128_percentile_raw"),
    },
    {
        "enabled": RUN_STAGE8_PERCENTILE_128_PREPROCESSED,
        "label": "Run optional Stage 8 Percentile 128 preprocessed candidate in foreground",
        "command": stage8_ablation_command("--candidate-tags", "s128_percentile_preprocessed"),
    },
    {
        "enabled": RUN_STAGE8_PERCENTILE_256_RAW,
        "label": "Run optional Stage 8 Percentile 256 raw candidate in foreground",
        "command": stage8_ablation_command("--candidate-tags", "s256_percentile_raw"),
    },
    {
        "enabled": RUN_STAGE8_PERCENTILE_256_PREPROCESSED,
        "label": "Run optional Stage 8 Percentile 256 preprocessed candidate in foreground",
        "command": stage8_ablation_command("--candidate-tags", "s256_percentile_preprocessed"),
    },
    {
        "enabled": RUN_STAGE8_FINALIZE_BEST,
        "label": "Finalize best Stage 8 candidate and evaluate it on test",
        "command": python_script(
            "scripts/run_stage8_quantization_ablation.py",
            *STAGE8_COMMON_ABLATION_ARGS,
            "--finalize-only",
            "--evaluate-test-best",
        ),
    },
    {
        "enabled": RUN_STUDENT_MODEL_COMPARISON,
        "label": "Refresh LoRA comparison after Stage 8",
        "command": python_script(
            "scripts/compare_student_models.py",
            "--output", REPO_ROOT / "results" / "tables" / "student_lora_model_comparison.csv",
            "--fp32-distilled-model-path", FP32_ONNX_PATH,
            "--int8-model-path", INT8_ONNX_PATH,
            "--teacher-summary-candidates",
            REPO_ROOT / "results" / "tables" / "lora_teacher_kvasir_seg_validation_clean_summary.csv",
            "--distilled-summary-candidates",
            REPO_ROOT / "results" / "tables" / f"{STAGE8_FP32_ONNX_MODEL_NAME}_validation_onnx_cpu_eval_summary.csv",
            REPO_ROOT / "results" / "tables" / f"{STAGE8_STUDENT_MODEL_NAME}_summary.csv",
            "--quantized-summary-candidates",
            REPO_ROOT / "results" / "tables" / f"{STAGE8_INT8_MODEL_NAME}_validation_quantized_eval_summary.csv",
            REPO_ROOT / "results" / "tables" / f"{STAGE8_INT8_T085_MODEL_NAME}_validation_quantized_eval_t085_summary.csv",
        ),
    },
]

for item in stage_8_commands:
    run_project_command_foreground(item["command"], enabled=item["enabled"], label=item["label"])


## Stage 9: INT8 Rescue

This foreground-only section diagnoses INT8 damage, runs selective mixed-precision PTQ candidates, builds boundary-aware calibration manifests, and optionally runs QAT distillation if PTQ still misses the Dice-drop gate.


In [1]:
from pathlib import Path
import subprocess
import sys
from datetime import datetime

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))


if "run_project_command_foreground" not in globals():
    def run_project_command_foreground(command, *, enabled=False, label=None):
        display_label = label or " ".join(str(part) for part in command)
        timestamp = datetime.now().strftime("%H:%M:%S")
        print(f"[{timestamp}] {display_label}")
        print("Command:", " ".join(str(part) for part in command))
        if not enabled:
            print("Skipped. Set the matching RUN_* variable to True to execute this cell.")
            return None
        process = subprocess.Popen(
            command,
            cwd=REPO_ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
        return_code = process.wait()
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, command)
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Completed: {display_label}")
        return return_code

if "python_script" not in globals():
    def python_script(script_name, *args):
        return [sys.executable, str(REPO_ROOT / script_name), *map(str, args)]

STAGE9_QAT_SEED = 6
RUN_STAGE9_INT8_RESCUE = True
RUN_STAGE9_DIAGNOSTICS = False
RUN_STAGE9_BOUNDARY_CALIBRATION = False
RUN_STAGE9_SELECTIVE_PTQ = False
RUN_STAGE9_QAT_DISTILLATION = True
RUN_STAGE9_REUSE_QAT_CHECKPOINT = False
RUN_STAGE9_FINAL_COMPARISON = True

STAGE9_CALIBRATION_SAMPLES = 64
STAGE9_DEBUG_SAMPLES = 8
STAGE9_EXCLUSION_RECIPES = ["none", "out_head_fp32", "last_decoder_fp32"]
STAGE9_CALIBRATION_STRATEGIES = ["representative", "boundary_hard"]
STAGE9_CALIBRATION_METHODS = ["minmax"]
STAGE9_PREPROCESSING_MODES = ["preprocess"]

STAGE9_STUDENT_MODEL_NAME = "student_distilled_lora_teacher"
STAGE9_MODEL_DIR = REPO_ROOT / "results" / "models" / STAGE9_STUDENT_MODEL_NAME
STAGE9_CHECKPOINT = STAGE9_MODEL_DIR / "best.pt"
STAGE9_TRAIN_MANIFEST = REPO_ROOT / "results" / "raw" / "lora_teacher_distillation_manifest_kvasir_seg_train_clean.jsonl"
STAGE9_VALIDATION_MANIFEST = REPO_ROOT / "results" / "raw" / "lora_teacher_distillation_manifest_kvasir_seg_validation_clean.jsonl"
STAGE9_TEST_MANIFEST = REPO_ROOT / "results" / "raw" / "lora_teacher_distillation_manifest_kvasir_seg_test_clean.jsonl"

def stage9_bool_flag(flag_name, enabled):
    return [f"--{flag_name}" if enabled else f"--no-{flag_name}"]

stage9_command = python_script(
    "scripts/run_stage9_int8_rescue.py",
    "--checkpoint", STAGE9_CHECKPOINT,
    "--model-dir", STAGE9_MODEL_DIR,
    "--train-manifest", STAGE9_TRAIN_MANIFEST,
    "--validation-manifest", STAGE9_VALIDATION_MANIFEST,
    "--test-manifest", STAGE9_TEST_MANIFEST,
    *stage9_bool_flag("run-diagnostics", RUN_STAGE9_DIAGNOSTICS),
    *stage9_bool_flag("run-boundary-calibration", RUN_STAGE9_BOUNDARY_CALIBRATION),
    *stage9_bool_flag("run-selective-ptq", RUN_STAGE9_SELECTIVE_PTQ),
    *stage9_bool_flag("run-qat", RUN_STAGE9_QAT_DISTILLATION),
    *( ["--reuse-qat-checkpoint"] if RUN_STAGE9_REUSE_QAT_CHECKPOINT else [] ),
    *stage9_bool_flag("run-final-comparison", RUN_STAGE9_FINAL_COMPARISON),
    "--debug-samples", STAGE9_DEBUG_SAMPLES,
    "--calibration-samples", STAGE9_CALIBRATION_SAMPLES,
    "--calibration-strategies", *STAGE9_CALIBRATION_STRATEGIES,
    "--calibration-methods", *STAGE9_CALIBRATION_METHODS,
    "--preprocessing-modes", *STAGE9_PREPROCESSING_MODES,
    "--exclusion-recipes", *STAGE9_EXCLUSION_RECIPES,
    "--qat-seed", STAGE9_QAT_SEED,
    "--batch-size", 1,
)

run_project_command_foreground(
    stage9_command,
    enabled=RUN_STAGE9_INT8_RESCUE,
    label="Run Stage 9 INT8 rescue pipeline in foreground",
)


[23:17:45] Run Stage 9 INT8 rescue pipeline in foreground
Command: /mnt/e/automations/Deployable MedSAM/.venv/bin/python /mnt/e/automations/Deployable MedSAM/scripts/run_stage9_int8_rescue.py --checkpoint /mnt/e/automations/Deployable MedSAM/results/models/student_distilled_lora_teacher/best.pt --model-dir /mnt/e/automations/Deployable MedSAM/results/models/student_distilled_lora_teacher --train-manifest /mnt/e/automations/Deployable MedSAM/results/raw/lora_teacher_distillation_manifest_kvasir_seg_train_clean.jsonl --validation-manifest /mnt/e/automations/Deployable MedSAM/results/raw/lora_teacher_distillation_manifest_kvasir_seg_validation_clean.jsonl --test-manifest /mnt/e/automations/Deployable MedSAM/results/raw/lora_teacher_distillation_manifest_kvasir_seg_test_clean.jsonl --no-run-diagnostics --no-run-boundary-calibration --no-run-selective-ptq --run-qat --run-final-comparison --debug-samples 8 --calibration-samples 64 --calibration-strategies representative boundary_hard --calib

0

In [2]:
import pandas as pd

STAGE9_TABLES = {
    "node_inventory": REPO_ROOT / "results" / "tables" / "lora_onnx_node_inventory.csv",
    "quantization_error": REPO_ROOT / "results" / "tables" / "lora_int8_quantization_error.csv",
    "selective_ptq_validation": REPO_ROOT / "results" / "tables" / "stage9_selective_ptq_validation.csv",
    "selective_ptq_best": REPO_ROOT / "results" / "tables" / "stage9_selective_ptq_best.csv",
    "final_comparison": REPO_ROOT / "results" / "tables" / "stage9_int8_rescue_final_comparison.csv",
}

for label, table_path in STAGE9_TABLES.items():
    if table_path.exists():
        print(label, "->", table_path)
        display(pd.read_csv(table_path).head(12))
    else:
        print(label, "missing:", table_path)


node_inventory -> /mnt/e/automations/Deployable MedSAM/results/tables/lora_onnx_node_inventory.csv


,node_index,node_name,node_display_name,op_type,conv_index,block_hint,input_names,output_names,exclusion_recipes
0,0,/in_conv/layers/layers.0/Conv,/in_conv/layers/layers.0/Conv,Conv,0.0,in_conv,image|onnx::Conv_200|onnx::Conv_201,/in_conv/layers/layers.0/Conv_output_0,first_last_fp32
1,1,/in_conv/layers/layers.2/Relu,/in_conv/layers/layers.2/Relu,Relu,NaN,in_conv,/in_conv/layers/layers.0/Conv_output_0,/in_conv/layers/layers.2/Relu_output_0,NaN
2,2,/in_conv/layers/layers.3/Conv,/in_conv/layers/layers.3/Conv,Conv,1.0,in_conv,/in_conv/layers/layers.2/Relu_output_0|onnx::C...,/in_conv/layers/layers.3/Conv_output_0,first_last_fp32
3,3,/in_conv/layers/layers.5/Relu,/in_conv/layers/layers.5/Relu,Relu,NaN,in_conv,/in_conv/layers/layers.3/Conv_output_0,/in_conv/layers/layers.5/Relu_output_0,NaN
4,4,/down1/layers/layers.0/MaxPool,/down1/layers/layers.0/MaxPool,MaxPool,NaN,down1,/in_conv/layers/layers.5/Relu_output_0,/down1/layers/layers.0/MaxPool_output_0,NaN
5,5,/down1/layers/layers.1/layers/layers.0/Conv,/down1/layers/layers.1/layers/layers.0/Conv,Conv,2.0,down1,/down1/layers/layers.0/MaxPool_output_0|onnx::...,/down1/layers/layers.1/layers/layers.0/Conv_ou...,NaN
6,6,/down1/layers/layers.1/layers/layers.2/Relu,/down1/layers/layers.1/layers/layers.2/Relu,Relu,NaN,down1,/down1/layers/layers.1/layers/layers.0/Conv_ou...,/down1/layers/layers.1/layers/layers.2/Relu_ou...,NaN
7,7,/down1/layers/layers.1/layers/layers.3/Conv,/down1/layers/layers.1/layers/layers.3/Conv,Conv,3.0,down1,/down1/layers/layers.1/layers/layers.2/Relu_ou...,/down1/layers/layers.1/layers/layers.3/Conv_ou...,NaN
8,8,/down1/layers/layers.1/layers/layers.5/Relu,/down1/layers/layers.1/layers/layers.5/Relu,Relu,NaN,down1,/down1/layers/layers.1/layers/layers.3/Conv_ou...,/down1/layers/layers.1/layers/layers.5/Relu_ou...,NaN
9,9,/down2/layers/layers.0/MaxPool,/down2/layers/layers.0/MaxPool,MaxPool,NaN,down2,/down1/layers/layers.1/layers/layers.5/Relu_ou...,/down2/layers/layers.0/MaxPool_output_0,NaN


quantization_error -> /mnt/e/automations/Deployable MedSAM/results/tables/lora_int8_quantization_error.csv


,status,tensor_name,comparison,value_count,mean_abs_error,max_abs_error,rmse,fp32_mean,int8_mean,note
0,available,logits,float_vs_post_qdq,524288,0.292282,3.903865,0.431308,-4.855390,-4.827607,NaN
1,available,/up3/conv/layers/layers.3/Conv_output_0,float_vs_post_qdq,8388608,0.149799,3.659726,0.249756,0.335868,0.330122,NaN
2,available,/up2/conv/layers/layers.3/Conv_output_0,float_vs_post_qdq,4194304,0.091465,2.043627,0.133792,-0.015659,-0.020062,NaN
3,available,/up3/conv/layers/layers.5/Relu_output_0,float_vs_post_qdq,8388608,0.088535,3.029818,0.192445,0.745994,0.745798,NaN
4,available,/up3/conv/layers/layers.0/Conv_output_0,float_vs_post_qdq,8388608,0.087756,2.659175,0.145761,-0.030144,-0.029282,NaN
5,available,/down3/layers/layers.1/layers/layers.3/Conv_ou...,float_vs_post_qdq,1048576,0.087543,1.016365,0.117108,-0.139972,-0.136808,NaN
6,available,/up1/conv/layers/layers.3/Conv_output_0,float_vs_post_qdq,2097152,0.085995,1.354619,0.118799,-0.073860,-0.077631,NaN
7,available,/up1/conv/layers/layers.0/Conv_output_0,float_vs_post_qdq,2097152,0.084220,1.160249,0.115717,-0.106192,-0.104163,NaN
8,available,/up2/conv/layers/layers.0/Conv_output_0,float_vs_post_qdq,4194304,0.082031,1.603027,0.116864,-0.029228,-0.025299,NaN
9,available,/down2/layers/layers.1/layers/layers.3/Conv_ou...,float_vs_post_qdq,2097152,0.081526,0.945068,0.110584,-0.201177,-0.207420,NaN


selective_ptq_validation -> /mnt/e/automations/Deployable MedSAM/results/tables/stage9_selective_ptq_validation.csv


,candidate_tag,status,exclusion_recipe,excluded_nodes,calibration_manifest_strategy,calibration_manifest,calibration_samples,requested_calibration_method,calibration_method,preprocessing_applied,...,validation_mean_binary_ece_roi,validation_mean_binary_ece_boundary,validation_mean_latency_ms,int8_model_size_mb,compression_ratio_vs_fp32,int8_onnx_path,validation_raw_path,validation_summary_path,threshold_sweep_path,error
0,s64_minmax_preprocessed_boundary_hard,available,none,NaN,boundary_hard,/mnt/e/automations/Deployable MedSAM/results/r...,64,minmax,MinMax,True,...,0.092522,0.303568,24.081407,0.511593,3.658850,/mnt/e/automations/Deployable MedSAM/results/m...,/mnt/e/automations/Deployable MedSAM/results/r...,/mnt/e/automations/Deployable MedSAM/results/t...,/mnt/e/automations/Deployable MedSAM/results/t...,NaN
1,s64_minmax_preprocessed_boundary_hard_out_head...,available,out_head_fp32,/out_conv/Conv,boundary_hard,/mnt/e/automations/Deployable MedSAM/results/r...,64,minmax,MinMax,True,...,0.091605,0.302357,26.290600,0.510135,3.669307,/mnt/e/automations/Deployable MedSAM/results/m...,/mnt/e/automations/Deployable MedSAM/results/r...,/mnt/e/automations/Deployable MedSAM/results/t...,/mnt/e/automations/Deployable MedSAM/results/t...,NaN
2,s64_minmax_preprocessed_boundary_hard_last_dec...,available,last_decoder_fp32,/up3/conv/layers/layers.0/Conv|/up3/conv/layer...,boundary_hard,/mnt/e/automations/Deployable MedSAM/results/r...,64,minmax,MinMax,True,...,0.089326,0.291966,28.585100,0.532919,3.512432,/mnt/e/automations/Deployable MedSAM/results/m...,/mnt/e/automations/Deployable MedSAM/results/r...,/mnt/e/automations/Deployable MedSAM/results/t...,/mnt/e/automations/Deployable MedSAM/results/t...,NaN


selective_ptq_best -> /mnt/e/automations/Deployable MedSAM/results/tables/stage9_selective_ptq_best.csv


,candidate_tag,status,exclusion_recipe,excluded_nodes,calibration_manifest_strategy,calibration_manifest,qat_seed,best_threshold,validation_mean_dice,validation_mean_iou,validation_mean_precision,validation_mean_recall,validation_mean_binary_ece_roi,validation_mean_binary_ece_boundary,validation_mean_latency_ms,int8_model_size_mb,compression_ratio_vs_fp32,int8_onnx_path,validation_summary_path,error
0,s64_minmax_preprocessed_last_decoder_fp32,available,last_decoder_fp32,/up3/conv/layers/layers.0/Conv|/up3/conv/layer...,representative,/mnt/e/automations/Deployable MedSAM/results/r...,NaN,0.825,0.760583,0.64886,0.807022,0.789119,0.088421,0.2897,25.339653,0.532918,3.512439,/mnt/e/automations/Deployable MedSAM/results/m...,/mnt/e/automations/Deployable MedSAM/results/t...,NaN


final_comparison -> /mnt/e/automations/Deployable MedSAM/results/tables/stage9_int8_rescue_final_comparison.csv


,label,status,summary_path,qat_seed,split,model_id,mean_dice,mean_iou,mean_binary_ece_roi,mean_binary_ece_boundary,mean_latency_ms,onnx_model_size_mb,compression_ratio_vs_fp32,threshold,dice_drop_vs_fp32,passes_dice_drop_gate_002
0,fp32_onnx_test,available,/mnt/e/automations/Deployable MedSAM/results/t...,NaN,test,student_distilled_lora_teacher_fp32_onnx,0.777909,0.666922,0.073204,0.160493,36.505787,1.871842,NaN,NaN,0.000000,True
1,naive_int8_test,available,/mnt/e/automations/Deployable MedSAM/results/t...,NaN,test,student_distilled_lora_teacher_int8,0.744729,0.624883,0.081175,0.163876,36.231593,0.512587,3.651755,NaN,0.033180,False
2,stage9_selective_ptq_best_test,available,/mnt/e/automations/Deployable MedSAM/results/t...,NaN,test,stage9_int8_s64_minmax_preprocessed_last_decod...,0.745919,0.629845,0.090272,0.293491,27.722727,0.532918,3.512439,0.825,0.031990,False
3,stage9_qat_int8_best_test,available,/mnt/e/automations/Deployable MedSAM/results/t...,6.0,test,stage9_int8_s64_minmax_preprocessed_qat_bounda...,0.772903,0.665667,0.075663,0.169788,33.699933,0.511594,3.658843,0.550,0.005006,True


In [3]:
from pathlib import Path
import sys

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd

STAGE8_STUDENT_MODEL_NAME = "student_distilled_lora_teacher"
STAGE8_FP32_ONNX_MODEL_NAME = f"{STAGE8_STUDENT_MODEL_NAME}_fp32_onnx"
STAGE8_INT8_MODEL_NAME = f"{STAGE8_STUDENT_MODEL_NAME}_int8"
STAGE8_INT8_T085_MODEL_NAME = f"{STAGE8_INT8_MODEL_NAME}_t085"

STAGE8_TABLES = {
    "fp32_onnx_validation": REPO_ROOT / "results" / "tables" / f"{STAGE8_FP32_ONNX_MODEL_NAME}_validation_onnx_cpu_eval_summary.csv",
    "fp32_onnx_test": REPO_ROOT / "results" / "tables" / f"{STAGE8_FP32_ONNX_MODEL_NAME}_test_onnx_cpu_eval_summary.csv",
    "int8_t085_validation": REPO_ROOT / "results" / "tables" / f"{STAGE8_INT8_T085_MODEL_NAME}_validation_quantized_eval_t085_summary.csv",
    "int8_t085_test": REPO_ROOT / "results" / "tables" / f"{STAGE8_INT8_T085_MODEL_NAME}_test_quantized_eval_t085_summary.csv",
    "threshold_sweep": REPO_ROOT / "results" / "tables" / f"{STAGE8_INT8_MODEL_NAME}_validation_threshold_sweep.csv",
    "ablation": REPO_ROOT / "results" / "tables" / "stage8_lora_quantization_ablation_validation.csv",
    "best": REPO_ROOT / "results" / "tables" / "stage8_lora_quantization_ablation_best.csv",
    "comparison": REPO_ROOT / "results" / "tables" / "student_lora_model_comparison.csv",
}

summary_rows = []
for label, path in STAGE8_TABLES.items():
    if label in {"threshold_sweep", "ablation", "best", "comparison"}:
        continue
    if path.exists():
        df = pd.read_csv(path)
        if not df.empty:
            row = df.iloc[0].to_dict()
            row["label"] = label
            row["source"] = path.name
            summary_rows.append(row)

if summary_rows:
    stage8_summary_df = pd.DataFrame(summary_rows)
    display_columns = [
        "label", "source", "split", "model_id", "sample_count", "threshold", "mean_dice", "mean_iou",
        "mean_precision", "mean_recall", "mean_binary_ece_roi", "mean_binary_ece_boundary",
        "mean_latency_ms", "onnx_model_size_mb", "compression_ratio_vs_fp32", "preprocessing_applied",
    ]
    display(stage8_summary_df[[column for column in display_columns if column in stage8_summary_df.columns]])
else:
    print("No Stage 8 FP32/threshold summary outputs are available yet.")

if STAGE8_TABLES["threshold_sweep"].exists():
    print("Top threshold candidates")
    display(pd.read_csv(STAGE8_TABLES["threshold_sweep"]).head(10))

if STAGE8_TABLES["ablation"].exists():
    ablation_df = pd.read_csv(STAGE8_TABLES["ablation"])
    display_columns = [
        "candidate_tag", "status", "calibration_samples", "requested_calibration_method", "calibration_method",
        "preprocessing_applied", "best_threshold", "validation_mean_dice", "validation_mean_iou",
        "validation_mean_precision", "validation_mean_recall", "validation_mean_binary_ece_roi",
        "validation_mean_binary_ece_boundary", "validation_mean_latency_ms", "compression_ratio_vs_fp32",
    ]
    display(ablation_df[[column for column in display_columns if column in ablation_df.columns]].sort_values("validation_mean_dice", ascending=False))

if STAGE8_TABLES["best"].exists():
    print("Selected Stage 8 best candidate")
    display(pd.read_csv(STAGE8_TABLES["best"]))

if STAGE8_TABLES["comparison"].exists():
    print("Current comparison table")
    display(pd.read_csv(STAGE8_TABLES["comparison"]))


,label,source,split,model_id,sample_count,mean_dice,mean_iou,mean_precision,mean_recall,mean_binary_ece_roi,mean_binary_ece_boundary,mean_latency_ms,onnx_model_size_mb,compression_ratio_vs_fp32,preprocessing_applied
0,fp32_onnx_validation,student_distilled_lora_teacher_fp32_onnx_valid...,validation,student_distilled_lora_teacher_fp32_onnx,150,0.768833,0.658761,0.775088,0.845552,0.071536,0.162947,37.395527,1.871842,NaN,NaN
1,fp32_onnx_test,student_distilled_lora_teacher_fp32_onnx_test_...,test,student_distilled_lora_teacher_fp32_onnx,150,0.777909,0.666922,0.800018,0.817860,0.073204,0.160493,36.505787,1.871842,NaN,NaN
2,int8_t085_validation,student_distilled_lora_teacher_int8_t085_valid...,validation,student_distilled_lora_teacher_int8_t085,150,0.757700,0.645591,0.804915,0.779842,0.093380,0.305436,34.853693,0.512587,3.651755,False
3,int8_t085_test,student_distilled_lora_teacher_int8_t085_test_...,test,student_distilled_lora_teacher_int8_t085,150,0.741437,0.624897,0.827551,0.738138,0.093870,0.308962,33.082120,0.512587,3.651755,False


Top threshold candidates


,label,threshold,sample_count,mean_dice,median_dice,mean_iou,mean_precision,mean_recall,samples_dice_below_0_5,samples_dice_below_0_3
0,lora_stage7_int8_minmax_s128,0.825,150,0.757775,0.833131,0.645663,0.798403,0.786451,17,7
1,lora_stage7_int8_minmax_s128,0.850,150,0.757700,0.832432,0.645591,0.804915,0.779842,17,7
2,lora_stage7_int8_minmax_s128,0.800,150,0.757578,0.832975,0.645445,0.791855,0.792913,15,7
3,lora_stage7_int8_minmax_s128,0.775,150,0.757244,0.829601,0.645009,0.785557,0.798879,14,8
4,lora_stage7_int8_minmax_s128,0.875,150,0.757203,0.830565,0.645055,0.811804,0.772486,17,7
5,lora_stage7_int8_minmax_s128,0.750,150,0.756555,0.826764,0.644159,0.779171,0.804507,14,8
6,lora_stage7_int8_minmax_s128,0.900,150,0.756283,0.826805,0.644107,0.818979,0.764430,18,7
7,lora_stage7_int8_minmax_s128,0.700,150,0.755860,0.817221,0.643296,0.772882,0.810175,14,8
8,lora_stage7_int8_minmax_s128,0.725,150,0.755860,0.817221,0.643296,0.772882,0.810175,14,8
9,lora_stage7_int8_minmax_s128,0.675,150,0.754854,0.812903,0.642070,0.766420,0.815524,13,8


,candidate_tag,status,calibration_samples,requested_calibration_method,calibration_method,preprocessing_applied,best_threshold,validation_mean_dice,validation_mean_iou,validation_mean_precision,validation_mean_recall,validation_mean_binary_ece_roi,validation_mean_binary_ece_boundary,validation_mean_latency_ms,compression_ratio_vs_fp32
3,s64_minmax_preprocessed,available,64,minmax,MinMax,True,0.825,0.760484,0.648730,0.813231,0.789350,0.090008,0.296745,30.488153,3.658857
2,s64_minmax_raw,available,64,minmax,MinMax,False,0.825,0.760484,0.648730,0.813231,0.789350,0.090008,0.296745,34.496793,3.651762
1,s32_minmax_preprocessed,available,32,minmax,MinMax,True,0.825,0.758665,0.647277,0.807020,0.786196,0.091428,0.295191,35.172760,3.658864
0,s32_minmax_raw,available,32,minmax,MinMax,False,0.825,0.758665,0.647277,0.807020,0.786196,0.091428,0.295191,30.969433,3.651769
7,s256_minmax_preprocessed,available,256,minmax,MinMax,True,0.850,0.758361,0.646109,0.805094,0.787752,0.092165,0.304343,32.102433,3.658843
6,s256_minmax_raw,available,256,minmax,MinMax,False,0.850,0.758361,0.646109,0.805094,0.787752,0.092165,0.304343,31.998760,3.651748
5,s128_minmax_preprocessed,available,128,minmax,MinMax,True,0.825,0.757775,0.645663,0.798403,0.786451,0.092209,0.298056,33.614327,3.658850
4,s128_minmax_raw,available,128,minmax,MinMax,False,0.825,0.757775,0.645663,0.798403,0.786451,0.092209,0.298056,32.522620,3.651755
11,s64_percentile_preprocessed,available,64,percentile,Percentile,True,0.800,0.757691,0.646048,0.801774,0.790045,0.090643,0.277883,32.182067,3.658807
10,s64_percentile_raw,available,64,percentile,Percentile,False,0.800,0.757691,0.646048,0.801774,0.790045,0.090643,0.277883,34.263520,3.651719


Selected Stage 8 best candidate


,candidate_tag,status,calibration_samples,requested_calibration_method,calibration_method,preprocessing_applied,preprocessing_status,best_threshold,validation_mean_dice,validation_mean_iou,...,validation_mean_binary_ece_roi,validation_mean_binary_ece_boundary,validation_mean_latency_ms,int8_model_size_mb,compression_ratio_vs_fp32,int8_onnx_path,validation_raw_path,validation_summary_path,threshold_sweep_path,error
0,s64_minmax_preprocessed,available,64,minmax,MinMax,True,applied,0.825,0.760484,0.64873,...,0.090008,0.296745,30.488153,0.511592,3.658857,/mnt/e/automations/Deployable MedSAM/results/m...,/mnt/e/automations/Deployable MedSAM/results/r...,/mnt/e/automations/Deployable MedSAM/results/t...,/mnt/e/automations/Deployable MedSAM/results/t...,NaN


Current comparison table


,model,status,source,sample_count,trainable_parameters,mean_dice,mean_iou,mean_precision,mean_recall,mean_brier_full,...,mean_binary_ece_foreground,mean_binary_ece_roi,mean_binary_ece_boundary,mean_latency_ms,model_file_size_mb,fp32_reference_model_file_size_mb,compression_ratio_vs_fp32,speedup_vs_teacher,speedup_vs_fp32_distilled,dice_drop_vs_fp32_distilled
0,MedSAM teacher clean,available,lora_teacher_kvasir_seg_validation_clean_summa...,150,not_available,0.939764,0.893216,0.941966,0.942385,0.013611,...,0.030762,0.024716,0.080790,285.295060,not_available,not_available,not_available,1.000000,not_available,not_available
1,GT-only U-Net baseline,available,student_baseline_summary.csv,150,488001,0.738322,0.623698,0.796289,0.770716,0.050347,...,0.167383,0.089152,0.165073,30.249360,not_available,not_available,not_available,9.431441,not_available,not_available
2,Teacher-distilled U-Net,available,student_distilled_lora_teacher_fp32_onnx_valid...,150,488001,0.768833,0.658761,0.775088,0.845552,0.047590,...,0.113416,0.071536,0.162947,37.395527,1.871842,not_available,not_available,7.629123,1.0,0.0
3,Quantized/distilled U-Net,available,student_distilled_lora_teacher_int8_validation...,150,488001,0.748342,0.634276,0.747333,0.841126,0.051488,...,0.119001,0.076936,0.165510,37.273340,0.512587,1.871842,3.651755,7.654132,1.003278,0.020491


## Stage 10: Stronger Baselines and CPU Deployment Timing

This section fills two study completeness checks after Stage 9 is complete. It can evaluate the GT-only U-Net on the Kvasir test split, rebuild the stronger baseline table, and run a CPU-only ONNX Runtime benchmark with warmup and repeated measurements. Keep the CPU benchmark off while training is running.

In [3]:
from pathlib import Path
import subprocess
import sys
from datetime import datetime

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))

if "run_project_command_foreground" not in globals():
    def run_project_command_foreground(command, *, enabled=False, label=None):
        display_label = label or " ".join(str(part) for part in command)
        timestamp = datetime.now().strftime("%H:%M:%S")
        print(f"[{timestamp}] {display_label}")
        print("Command:", " ".join(str(part) for part in command))
        if not enabled:
            print("Skipped. Set the matching RUN_* variable to True to execute this cell.")
            return None
        process = subprocess.Popen(
            command,
            cwd=REPO_ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
        return_code = process.wait()
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, command)
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Completed: {display_label}")
        return return_code

if "python_script" not in globals():
    def python_script(script_name, *args):
        return [sys.executable, str(REPO_ROOT / script_name), *map(str, args)]

RUN_GT_ONLY_BASELINE_TEST_EVAL = False
RUN_Q1_BASELINE_TABLE_BUILD = False
RUN_ONNX_CPU_DEPLOYMENT_BENCHMARK = True

CPU_BENCHMARK_SAMPLE_LIMIT = 150
CPU_BENCHMARK_BATCH_SIZE = 1
CPU_BENCHMARK_WARMUP_RUNS = 2
CPU_BENCHMARK_REPEAT_RUNS = 5
CPU_BENCHMARK_INTRA_THREADS = 1
CPU_BENCHMARK_INTER_THREADS = 1

stage10_tasks = [
    {
        "enabled": RUN_GT_ONLY_BASELINE_TEST_EVAL,
        "label": "Evaluate GT-only FP32 U-Net baseline on Kvasir test",
        "command": python_script(
            "scripts/evaluate_student_model.py",
            "--split", "test",
            "--checkpoint", REPO_ROOT / "results" / "models" / "student_baseline" / "best.pt",
            "--model-name", "student_baseline",
            "--run-tag", "test_eval",
            "--device", "auto",
            "--batch-size", 8,
        ),
    },
    {
        "enabled": RUN_Q1_BASELINE_TABLE_BUILD,
        "label": "Build stronger Q1 baseline table",
        "command": python_script("scripts/build_q1_baseline_table.py"),
    },
    {
        "enabled": RUN_ONNX_CPU_DEPLOYMENT_BENCHMARK,
        "label": "Benchmark ONNX deployment models on CPU",
        "command": python_script(
            "scripts/benchmark_onnx_cpu_latency.py",
            "--sample-limit", CPU_BENCHMARK_SAMPLE_LIMIT,
            "--batch-size", CPU_BENCHMARK_BATCH_SIZE,
            "--warmup-runs", CPU_BENCHMARK_WARMUP_RUNS,
            "--repeat-runs", CPU_BENCHMARK_REPEAT_RUNS,
            "--intra-op-num-threads", CPU_BENCHMARK_INTRA_THREADS,
            "--inter-op-num-threads", CPU_BENCHMARK_INTER_THREADS,
        ),
    },
]

for task in stage10_tasks:
    run_project_command_foreground(task["command"], enabled=task["enabled"], label=task["label"])


[23:57:53] Evaluate GT-only FP32 U-Net baseline on Kvasir test
Command: /mnt/e/automations/Deployable MedSAM/.venv/bin/python /mnt/e/automations/Deployable MedSAM/scripts/evaluate_student_model.py --split test --checkpoint /mnt/e/automations/Deployable MedSAM/results/models/student_baseline/best.pt --model-name student_baseline --run-tag test_eval --device auto --batch-size 8
Skipped. Set the matching RUN_* variable to True to execute this cell.
[23:57:53] Build stronger Q1 baseline table
Command: /mnt/e/automations/Deployable MedSAM/.venv/bin/python /mnt/e/automations/Deployable MedSAM/scripts/build_q1_baseline_table.py
Skipped. Set the matching RUN_* variable to True to execute this cell.
[23:57:53] Benchmark ONNX deployment models on CPU
Command: /mnt/e/automations/Deployable MedSAM/.venv/bin/python /mnt/e/automations/Deployable MedSAM/scripts/benchmark_onnx_cpu_latency.py --sample-limit 150 --batch-size 1 --warmup-runs 2 --repeat-runs 5 --intra-op-num-threads 1 --inter-op-num-threa

In [2]:
from pathlib import Path
import pandas as pd
import sys

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent

RESULT_TABLES = REPO_ROOT / "results" / "tables"
summary_tables = {
    "stronger_baseline": RESULT_TABLES / "q1_stronger_baseline_table.csv",
    "qat_seed_reproducibility": RESULT_TABLES / "q1_qat_seed_reproducibility_table.csv",
    "onnx_cpu_timing": RESULT_TABLES / "q1_onnx_cpu_deployment_timing.csv",
}

for label, path in summary_tables.items():
    print("\n", label, "->", path)
    if not path.exists():
        print("Missing. Run the Stage 10 task that creates this table.")
        continue
    frame = pd.read_csv(path)
    columns = [
        "display_label", "label", "status", "split", "sample_count", "mean_dice", "mean_iou",
        "mean_latency_ms", "std_latency_ms", "median_latency_ms", "p95_latency_ms",
        "device", "execution_provider", "onnx_model_size_mb", "threshold", "exclusion_recipe",
        "qat_seed", "dice_drop_vs_fp32_onnx", "passes_dice_drop_gate_002", "note",
    ]
    available = [column for column in columns if column in frame.columns]
    display(frame[available] if available else frame)



 stronger_baseline -> /mnt/e/automations/Deployable MedSAM/results/tables/q1_stronger_baseline_table.csv


,display_label,status,split,sample_count,mean_dice,mean_iou,mean_latency_ms,device,onnx_model_size_mb,threshold,exclusion_recipe,qat_seed,dice_drop_vs_fp32_onnx,passes_dice_drop_gate_002,note
0,"MedSAM teacher, clean prompt",available,test,150.0,0.863613,0.780167,257.550207,cuda,NaN,NaN,NaN,NaN,NaN,not_applicable,NaN
1,LoRA-adapted MedSAM teacher,available,test,150.0,0.936846,0.891136,276.045220,cuda,NaN,NaN,NaN,NaN,NaN,not_applicable,NaN
2,GT-only FP32 U-Net,available,test,150.0,0.730889,0.615427,25.697533,cuda,NaN,NaN,NaN,NaN,NaN,not_applicable,NaN
3,Teacher-distilled FP32 U-Net,available,test,150.0,0.707398,0.583328,25.305573,cuda,NaN,NaN,NaN,NaN,NaN,not_applicable,NaN
4,LoRA-teacher FP32 U-Net,available,test,150.0,0.777901,0.666914,33.588187,cuda,NaN,NaN,NaN,NaN,NaN,not_applicable,NaN
5,LoRA-student FP32 ONNX,available,test,150.0,0.777909,0.666922,36.505787,onnxruntime:CPUExecutionProvider,1.871842,NaN,NaN,NaN,NaN,not_applicable,NaN
6,Naive INT8 QDQ,available,test,150.0,0.744729,0.624883,36.231593,onnxruntime:CPUExecutionProvider,0.512587,NaN,NaN,NaN,0.033180,False,NaN
7,Selective PTQ INT8,available,test,150.0,0.745919,0.629845,27.722727,onnxruntime:CPUExecutionProvider,0.532918,0.825,last_decoder_fp32,NaN,0.031990,False,NaN
8,QAT INT8 seed 3,available,test,150.0,0.784489,0.677453,37.384487,NaN,0.532918,0.625,last_decoder_fp32,3,-0.006580,True,Seed 3 full model archive was not available; s...
9,QAT INT8 seed 4,available,test,150.0,0.783973,0.677332,41.604153,onnxruntime:CPUExecutionProvider,0.511592,0.550,none,4,-0.006064,True,NaN



 qat_seed_reproducibility -> /mnt/e/automations/Deployable MedSAM/results/tables/q1_qat_seed_reproducibility_table.csv


,display_label,status,split,sample_count,mean_dice,mean_iou,mean_latency_ms,device,onnx_model_size_mb,threshold,exclusion_recipe,qat_seed,dice_drop_vs_fp32_onnx,passes_dice_drop_gate_002,note
0,QAT INT8 seed 3,available,test,150.0,0.784489,0.677453,37.384487,NaN,0.532918,0.625,last_decoder_fp32,3,-0.006580,True,Seed 3 full model archive was not available; s...
1,QAT INT8 seed 4,available,test,150.0,0.783973,0.677332,41.604153,onnxruntime:CPUExecutionProvider,0.511592,0.550,none,4,-0.006064,True,NaN
2,QAT INT8 seed 5,available,test,150.0,0.782924,0.674260,30.967440,onnxruntime:CPUExecutionProvider,0.532920,0.650,last_decoder_fp32,5,-0.005015,True,NaN



 onnx_cpu_timing -> /mnt/e/automations/Deployable MedSAM/results/tables/q1_onnx_cpu_deployment_timing.csv


,label,status,mean_latency_ms,std_latency_ms,median_latency_ms,p95_latency_ms,execution_provider,onnx_model_size_mb,note
0,LoRA-student FP32 ONNX,available,65.178808,2.800579,64.444945,70.561313,CPUExecutionProvider,1.871842,NaN
1,Naive INT8 QDQ,available,46.672010,2.253313,46.155884,50.992976,CPUExecutionProvider,0.512587,NaN
2,Selective PTQ INT8,available,52.229883,2.299982,51.680925,56.000205,CPUExecutionProvider,0.532918,NaN
3,QAT INT8 seed 3,missing_model,NaN,NaN,NaN,NaN,CPUExecutionProvider,NaN,Model file was not available for CPU timing. T...
4,QAT INT8 seed 4,available,46.639204,2.658115,45.983079,51.256188,CPUExecutionProvider,0.511592,NaN
5,QAT INT8 seed 5,available,52.503581,3.386428,51.930546,56.241521,CPUExecutionProvider,0.532920,NaN


## Cross-Dataset Generalization: ISIC 2018 and BUSI

This section checks whether the Kvasir-trained LoRA-student still segments lesions on external medical datasets. It is an external validity test, not another training stage: ISIC tests skin lesion domain shift and BUSI tests ultrasound domain shift.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
from datetime import datetime

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))

if "python_script" not in globals():
    def python_script(script_name, *args):
        return [sys.executable, str(REPO_ROOT / script_name), *map(str, args)]

if "run_project_command_foreground" not in globals():
    def run_project_command_foreground(command, *, enabled=False, label=None):
        display_label = label or " ".join(str(part) for part in command)
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {display_label}")
        print("Command:", " ".join(str(part) for part in command))
        if not enabled:
            print("Skipped. Set the matching RUN_* variable to True to execute this cell.")
            return None
        completed = subprocess.run(command, cwd=REPO_ROOT)
        completed.check_returncode()
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Completed: {display_label}")
        return completed.returncode

RUN_EXTERNAL_DATASET_MANIFEST_PREP = False

external_manifest_command = python_script("scripts/prepare_external_segmentation_manifests.py")
run_project_command_foreground(
    external_manifest_command,
    enabled=RUN_EXTERNAL_DATASET_MANIFEST_PREP,
    label="Prepare ISIC/BUSI external split manifests",
)

EXTERNAL_SPLIT_MANIFESTS = {
    "isic_2018_task1": REPO_ROOT / "results" / "splits" / "isic_2018_task1_official_seed1.json",
    "busi": REPO_ROOT / "results" / "splits" / "busi_seed1.json",
}

for dataset_name, manifest_path in EXTERNAL_SPLIT_MANIFESTS.items():
    if manifest_path.exists():
        payload = json.loads(manifest_path.read_text())
        print(dataset_name, "counts=", payload["counts"], "manifest=", manifest_path)
    else:
        print(dataset_name, "manifest missing:", manifest_path)


[19:03:04] Prepare ISIC/BUSI external split manifests
Command: /mnt/e/automations/Deployable MedSAM/.venv/bin/python /mnt/e/automations/Deployable MedSAM/scripts/prepare_external_segmentation_manifests.py
Skipped. Set the matching RUN_* variable to True to execute this cell.
isic_2018_task1 counts= {'train': 2594, 'validation': 100, 'test': 1000} manifest= /mnt/e/automations/Deployable MedSAM/results/splits/isic_2018_task1_official_seed1.json
busi counts= {'train': 546, 'validation': 117, 'test': 117} manifest= /mnt/e/automations/Deployable MedSAM/results/splits/busi_seed1.json


In [2]:
RUN_CROSS_DATASET_TEST_EVAL = False

CROSS_DATASET_EVAL_SPLIT = "test"
CROSS_DATASET_DEVICE = "auto"
CROSS_DATASET_BATCH_SIZE = 8
CROSS_DATASET_NUM_WORKERS = 0

CROSS_DATASET_MODEL_SPECS = [
    {
        "enabled": True,
        "label": "lora_student_fp32",
        "checkpoint": REPO_ROOT / "results" / "models" / "student_distilled_lora_teacher" / "best.pt",
        "threshold": 0.5,
    },
    {
        "enabled": True,
        "label": "lora_student_qat_seed3",
        "checkpoint": REPO_ROOT / "results" / "models" / "student_distilled_lora_teacher_qat" / "best.pt",
        "threshold": 0.625,
    },
]

CROSS_DATASET_DATASETS = [
    {"dataset": "isic_2018_task1", "manifest": REPO_ROOT / "results" / "splits" / "isic_2018_task1_official_seed1.json"},
    {"dataset": "busi", "manifest": REPO_ROOT / "results" / "splits" / "busi_seed1.json"},
]

for model_spec in CROSS_DATASET_MODEL_SPECS:
    for dataset_spec in CROSS_DATASET_DATASETS:
        enabled = RUN_CROSS_DATASET_TEST_EVAL and model_spec["enabled"]
        dataset_name = dataset_spec["dataset"]
        model_name = f"{model_spec['label']}_cross_dataset"
        run_tag = f"{CROSS_DATASET_EVAL_SPLIT}_t{str(model_spec['threshold']).replace('.', '')}"
        command = python_script(
            "scripts/run_cross_dataset_student_eval.py",
            "--dataset", dataset_name,
            "--split-manifest", dataset_spec["manifest"],
            "--split", CROSS_DATASET_EVAL_SPLIT,
            "--checkpoint", model_spec["checkpoint"],
            "--model-name", model_name,
            "--run-tag", run_tag,
            "--threshold", model_spec["threshold"],
            "--device", CROSS_DATASET_DEVICE,
            "--batch-size", CROSS_DATASET_BATCH_SIZE,
            "--num-workers", CROSS_DATASET_NUM_WORKERS,
        )
        run_project_command_foreground(
            command,
            enabled=enabled,
            label=f"Cross-dataset eval: {model_spec['label']} on {dataset_name} {CROSS_DATASET_EVAL_SPLIT}",
        )


[19:03:30] Cross-dataset eval: lora_student_fp32 on isic_2018_task1 test
Command: /mnt/e/automations/Deployable MedSAM/.venv/bin/python /mnt/e/automations/Deployable MedSAM/scripts/run_cross_dataset_student_eval.py --dataset isic_2018_task1 --split-manifest /mnt/e/automations/Deployable MedSAM/results/splits/isic_2018_task1_official_seed1.json --split test --checkpoint /mnt/e/automations/Deployable MedSAM/results/models/student_distilled_lora_teacher/best.pt --model-name lora_student_fp32_cross_dataset --run-tag test_t05 --threshold 0.5 --device auto --batch-size 8 --num-workers 0
rows=1000
raw_output=/mnt/e/automations/Deployable MedSAM/results/raw/lora_student_fp32_cross_dataset_isic_2018_task1_test_t05.csv
summary_output=/mnt/e/automations/Deployable MedSAM/results/tables/lora_student_fp32_cross_dataset_isic_2018_task1_test_t05_summary.csv
[19:16:18] Completed: Cross-dataset eval: lora_student_fp32 on isic_2018_task1 test
[19:16:18] Cross-dataset eval: lora_student_fp32 on busi test

In [3]:
import pandas as pd

CROSS_DATASET_SUMMARY_TABLES = [
    REPO_ROOT / "results" / "tables" / "lora_student_fp32_cross_dataset_isic_2018_task1_test_t05_summary.csv",
    REPO_ROOT / "results" / "tables" / "lora_student_fp32_cross_dataset_busi_test_t05_summary.csv",
    REPO_ROOT / "results" / "tables" / "lora_student_qat_seed3_cross_dataset_isic_2018_task1_test_t0625_summary.csv",
    REPO_ROOT / "results" / "tables" / "lora_student_qat_seed3_cross_dataset_busi_test_t0625_summary.csv",
]

summary_rows = []
for table_path in CROSS_DATASET_SUMMARY_TABLES:
    if table_path.exists():
        frame = pd.read_csv(table_path)
        if not frame.empty:
            row = frame.iloc[0].to_dict()
            row["source"] = table_path.name
            summary_rows.append(row)
    else:
        print("Missing:", table_path)

if summary_rows:
    cross_dataset_summary = pd.DataFrame(summary_rows)
    display_columns = [
        "source", "dataset", "split", "model_id", "sample_count", "mean_dice", "mean_iou",
        "mean_precision", "mean_recall", "mean_binary_ece_roi", "mean_binary_ece_boundary",
        "mean_latency_ms", "source_checkpoint",
    ]
    display(cross_dataset_summary[[column for column in display_columns if column in cross_dataset_summary.columns]])
else:
    print("No cross-dataset summaries are available yet.")


,source,dataset,split,model_id,sample_count,mean_dice,mean_iou,mean_precision,mean_recall,mean_binary_ece_roi,mean_binary_ece_boundary,mean_latency_ms,source_checkpoint
0,lora_student_fp32_cross_dataset_isic_2018_task...,isic_2018_task1,test,lora_student_fp32_cross_dataset,1000,0.104154,0.065182,0.473530,0.074678,0.433094,0.417291,28.540264,/mnt/e/automations/Deployable MedSAM/results/m...
1,lora_student_fp32_cross_dataset_busi_test_t05_...,busi,test,lora_student_fp32_cross_dataset,117,0.059221,0.040890,0.095701,0.236179,0.158299,0.228917,31.955598,/mnt/e/automations/Deployable MedSAM/results/m...
2,lora_student_qat_seed3_cross_dataset_isic_2018...,isic_2018_task1,test,lora_student_qat_seed3_cross_dataset,1000,0.099850,0.062226,0.554910,0.065098,0.433354,0.444728,25.614312,/mnt/e/automations/Deployable MedSAM/results/m...
3,lora_student_qat_seed3_cross_dataset_busi_test...,busi,test,lora_student_qat_seed3_cross_dataset,117,0.069855,0.065024,0.110974,0.183876,0.197282,0.322330,42.128393,/mnt/e/automations/Deployable MedSAM/results/m...


### Cross-Dataset Threshold Sweep

The fixed thresholds showed weak zero-shot transfer on ISIC and BUSI. This sweep tests whether the external failures are mainly calibration problems: if Dice improves sharply at a lower threshold, the model still carries useful signal but is miscalibrated on the new domain; if it stays poor, domain adaptation is the next honest path.


In [ ]:
# CROSS_DATASET_THRESHOLD_SWEEP_CELL
from pathlib import Path
import subprocess
import sys
from datetime import datetime

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))

if "python_script" not in globals():
    def python_script(script_name, *args):
        return [sys.executable, str(REPO_ROOT / script_name), *map(str, args)]

if "run_project_command_foreground" not in globals():
    def run_project_command_foreground(command, *, enabled=False, label=None):
        display_label = label or " ".join(str(part) for part in command)
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {display_label}")
        print("Command:", " ".join(str(part) for part in command))
        if not enabled:
            print("Skipped. Set the matching RUN_* variable to True to execute this cell.")
            return None
        completed = subprocess.run(command, cwd=REPO_ROOT)
        completed.check_returncode()
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Completed: {display_label}")
        return completed.returncode

RUN_CROSS_DATASET_THRESHOLD_SWEEP = False

CROSS_DATASET_SWEEP_SPLIT = "test"
CROSS_DATASET_SWEEP_DEVICE = "auto"
CROSS_DATASET_SWEEP_BATCH_SIZE = 8
CROSS_DATASET_SWEEP_NUM_WORKERS = 0
CROSS_DATASET_SWEEP_THRESHOLDS = "0.025:0.95:0.025"

CROSS_DATASET_SWEEP_MODEL_SPECS = [
    {
        "enabled": True,
        "label": "lora_student_fp32",
        "checkpoint": REPO_ROOT / "results" / "models" / "student_distilled_lora_teacher" / "best.pt",
    },
    {
        "enabled": True,
        "label": "lora_student_qat_seed3",
        "checkpoint": REPO_ROOT / "results" / "models" / "student_distilled_lora_teacher_qat" / "best.pt",
    },
]

CROSS_DATASET_SWEEP_DATASETS = [
    {"dataset": "isic_2018_task1", "manifest": REPO_ROOT / "results" / "splits" / "isic_2018_task1_official_seed1.json"},
    {"dataset": "busi", "manifest": REPO_ROOT / "results" / "splits" / "busi_seed1.json"},
]

for model_spec in CROSS_DATASET_SWEEP_MODEL_SPECS:
    for dataset_spec in CROSS_DATASET_SWEEP_DATASETS:
        enabled = RUN_CROSS_DATASET_THRESHOLD_SWEEP and model_spec["enabled"]
        dataset_name = dataset_spec["dataset"]
        model_name = f"{model_spec['label']}_cross_dataset"
        run_tag = f"{CROSS_DATASET_SWEEP_SPLIT}_threshold_sweep"
        output_stem = f"{model_name}_{dataset_name}_{run_tag}"
        command = python_script(
            "scripts/run_cross_dataset_threshold_sweep.py",
            "--dataset", dataset_name,
            "--split-manifest", dataset_spec["manifest"],
            "--split", CROSS_DATASET_SWEEP_SPLIT,
            "--checkpoint", model_spec["checkpoint"],
            "--model-name", model_name,
            "--run-tag", run_tag,
            "--label", f"{model_spec['label']}_{dataset_name}_{CROSS_DATASET_SWEEP_SPLIT}",
            "--thresholds", CROSS_DATASET_SWEEP_THRESHOLDS,
            "--device", CROSS_DATASET_SWEEP_DEVICE,
            "--batch-size", CROSS_DATASET_SWEEP_BATCH_SIZE,
            "--num-workers", CROSS_DATASET_SWEEP_NUM_WORKERS,
            "--sweep-output", REPO_ROOT / "results" / "tables" / f"{output_stem}.csv",
            "--best-raw-output", REPO_ROOT / "results" / "raw" / f"{output_stem}_best_threshold.csv",
            "--best-summary-output", REPO_ROOT / "results" / "tables" / f"{output_stem}_best_threshold_summary.csv",
        )
        run_project_command_foreground(
            command,
            enabled=enabled,
            label=f"Cross-dataset threshold sweep: {model_spec['label']} on {dataset_name} {CROSS_DATASET_SWEEP_SPLIT}",
        )


In [ ]:
# CROSS_DATASET_THRESHOLD_SWEEP_RESULTS_CELL
from pathlib import Path
import pandas as pd
import sys

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))

sweep_tables = sorted((REPO_ROOT / "results" / "tables").glob("*_cross_dataset_*_test_threshold_sweep.csv"))
best_summary_tables = sorted((REPO_ROOT / "results" / "tables").glob("*_cross_dataset_*_test_threshold_sweep_best_threshold_summary.csv"))

sweep_best_rows = []
for table_path in sweep_tables:
    frame = pd.read_csv(table_path)
    if frame.empty:
        continue
    row = frame.iloc[0].to_dict()
    row["source"] = table_path.name
    sweep_best_rows.append(row)

if sweep_best_rows:
    sweep_best = pd.DataFrame(sweep_best_rows)
    sweep_columns = [
        "source", "dataset", "split", "model_id", "threshold", "sample_count",
        "mean_dice", "median_dice", "mean_iou", "mean_precision", "mean_recall",
        "samples_dice_below_0_5", "samples_dice_below_0_3", "mean_latency_ms",
    ]
    display(sweep_best[[column for column in sweep_columns if column in sweep_best.columns]])
else:
    print("No cross-dataset threshold sweep tables found yet.")

best_eval_rows = []
for table_path in best_summary_tables:
    frame = pd.read_csv(table_path)
    if frame.empty:
        continue
    row = frame.iloc[0].to_dict()
    row["source"] = table_path.name
    best_eval_rows.append(row)

if best_eval_rows:
    best_eval = pd.DataFrame(best_eval_rows)
    eval_columns = [
        "source", "dataset", "split", "model_id", "selected_threshold", "sample_count",
        "mean_dice", "mean_iou", "mean_precision", "mean_recall",
        "mean_binary_ece_roi", "mean_binary_ece_boundary", "mean_latency_ms", "source_checkpoint",
    ]
    display(best_eval[[column for column in eval_columns if column in best_eval.columns]])

for table_path in sweep_tables:
    frame = pd.read_csv(table_path)
    preview_columns = ["threshold", "mean_dice", "median_dice", "mean_iou", "mean_precision", "mean_recall"]
    print(f"Top threshold candidates: {table_path.name}")
    display(frame[[column for column in preview_columns if column in frame.columns]].head(8))


### Validation-Selected Cross-Dataset Thresholds

This is the honest external-dataset check: sweep thresholds on the validation split, then evaluate the test split once using the validation-selected threshold. The earlier test-set sweep stays useful as a diagnostic upper bound, but this section is the defensible result for reporting.


In [6]:
# CROSS_DATASET_VALIDATION_SELECTED_THRESHOLD_CELL
from pathlib import Path
import pandas as pd
import subprocess
import sys
from datetime import datetime

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))

if "python_script" not in globals():
    def python_script(script_name, *args):
        return [sys.executable, str(REPO_ROOT / script_name), *map(str, args)]

if "run_project_command_foreground" not in globals():
    def run_project_command_foreground(command, *, enabled=False, label=None):
        display_label = label or " ".join(str(part) for part in command)
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {display_label}")
        print("Command:", " ".join(str(part) for part in command))
        if not enabled:
            print("Skipped. Set the matching RUN_* variable to True to execute this cell.")
            return None
        completed = subprocess.run(command, cwd=REPO_ROOT)
        completed.check_returncode()
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Completed: {display_label}")
        return completed.returncode

RUN_CROSS_DATASET_VALIDATION_THRESHOLD_SWEEP = True
RUN_CROSS_DATASET_VALIDATION_SELECTED_TEST_EVAL = True

CROSS_DATASET_VALIDATION_SELECTION_DEVICE = "auto"
CROSS_DATASET_VALIDATION_SELECTION_BATCH_SIZE = 8
CROSS_DATASET_VALIDATION_SELECTION_NUM_WORKERS = 0
CROSS_DATASET_VALIDATION_SELECTION_THRESHOLDS = "0.025:0.95:0.025"

CROSS_DATASET_VALIDATION_MODEL_SPECS = [
    {
        "enabled": True,
        "label": "lora_student_fp32",
        "fixed_threshold": 0.5,
        "checkpoint": REPO_ROOT / "results" / "models" / "student_distilled_lora_teacher" / "best.pt",
    },
    {
        "enabled": True,
        "label": "lora_student_qat_seed3",
        "fixed_threshold": 0.625,
        "checkpoint": REPO_ROOT / "results" / "models" / "student_distilled_lora_teacher_qat" / "best.pt",
    },
]

CROSS_DATASET_VALIDATION_DATASETS = [
    {"dataset": "isic_2018_task1", "manifest": REPO_ROOT / "results" / "splits" / "isic_2018_task1_official_seed1.json"},
    {"dataset": "busi", "manifest": REPO_ROOT / "results" / "splits" / "busi_seed1.json"},
]

def threshold_tag(value):
    return f"t{str(round(float(value), 6)).replace('.', '')}"

for model_spec in CROSS_DATASET_VALIDATION_MODEL_SPECS:
    for dataset_spec in CROSS_DATASET_VALIDATION_DATASETS:
        if not model_spec["enabled"]:
            continue
        dataset_name = dataset_spec["dataset"]
        model_name = f"{model_spec['label']}_cross_dataset"
        output_stem = f"{model_name}_{dataset_name}_validation_threshold_sweep"
        validation_sweep_output = REPO_ROOT / "results" / "tables" / f"{output_stem}.csv"
        validation_best_raw_output = REPO_ROOT / "results" / "raw" / f"{output_stem}_best_threshold.csv"
        validation_best_summary_output = REPO_ROOT / "results" / "tables" / f"{output_stem}_best_threshold_summary.csv"

        validation_command = python_script(
            "scripts/run_cross_dataset_threshold_sweep.py",
            "--dataset", dataset_name,
            "--split-manifest", dataset_spec["manifest"],
            "--split", "validation",
            "--checkpoint", model_spec["checkpoint"],
            "--model-name", model_name,
            "--run-tag", "validation_threshold_sweep",
            "--label", f"{model_spec['label']}_{dataset_name}_validation",
            "--thresholds", CROSS_DATASET_VALIDATION_SELECTION_THRESHOLDS,
            "--device", CROSS_DATASET_VALIDATION_SELECTION_DEVICE,
            "--batch-size", CROSS_DATASET_VALIDATION_SELECTION_BATCH_SIZE,
            "--num-workers", CROSS_DATASET_VALIDATION_SELECTION_NUM_WORKERS,
            "--sweep-output", validation_sweep_output,
            "--best-raw-output", validation_best_raw_output,
            "--best-summary-output", validation_best_summary_output,
        )
        run_project_command_foreground(
            validation_command,
            enabled=RUN_CROSS_DATASET_VALIDATION_THRESHOLD_SWEEP,
            label=f"Validation threshold sweep: {model_spec['label']} on {dataset_name}",
        )

        if not RUN_CROSS_DATASET_VALIDATION_SELECTED_TEST_EVAL:
            continue
        if not validation_sweep_output.exists():
            print("Missing validation sweep table, cannot select threshold:", validation_sweep_output)
            continue
        validation_sweep = pd.read_csv(validation_sweep_output)
        if validation_sweep.empty:
            print("Empty validation sweep table, cannot select threshold:", validation_sweep_output)
            continue
        selected_threshold = float(validation_sweep.iloc[0]["threshold"])
        selected_tag = threshold_tag(selected_threshold)
        test_output_stem = f"{model_name}_{dataset_name}_test_validation_selected_threshold"
        test_command = python_script(
            "scripts/run_cross_dataset_student_eval.py",
            "--dataset", dataset_name,
            "--split-manifest", dataset_spec["manifest"],
            "--split", "test",
            "--checkpoint", model_spec["checkpoint"],
            "--model-name", model_name,
            "--run-tag", f"test_validation_selected_{selected_tag}",
            "--threshold", selected_threshold,
            "--device", CROSS_DATASET_VALIDATION_SELECTION_DEVICE,
            "--batch-size", CROSS_DATASET_VALIDATION_SELECTION_BATCH_SIZE,
            "--num-workers", CROSS_DATASET_VALIDATION_SELECTION_NUM_WORKERS,
            "--raw-output", REPO_ROOT / "results" / "raw" / f"{test_output_stem}.csv",
            "--summary-output", REPO_ROOT / "results" / "tables" / f"{test_output_stem}_summary.csv",
        )
        print(f"Selected validation threshold for {model_spec['label']} on {dataset_name}: {selected_threshold}")
        run_project_command_foreground(
            test_command,
            enabled=True,
            label=f"Test eval with validation-selected threshold: {model_spec['label']} on {dataset_name}",
        )


[21:36:24] Validation threshold sweep: lora_student_fp32 on isic_2018_task1
Command: /mnt/e/automations/Deployable MedSAM/.venv/bin/python /mnt/e/automations/Deployable MedSAM/scripts/run_cross_dataset_threshold_sweep.py --dataset isic_2018_task1 --split-manifest /mnt/e/automations/Deployable MedSAM/results/splits/isic_2018_task1_official_seed1.json --split validation --checkpoint /mnt/e/automations/Deployable MedSAM/results/models/student_distilled_lora_teacher/best.pt --model-name lora_student_fp32_cross_dataset --run-tag validation_threshold_sweep --label lora_student_fp32_isic_2018_task1_validation --thresholds 0.025:0.95:0.025 --device auto --batch-size 8 --num-workers 0 --sweep-output /mnt/e/automations/Deployable MedSAM/results/tables/lora_student_fp32_cross_dataset_isic_2018_task1_validation_threshold_sweep.csv --best-raw-output /mnt/e/automations/Deployable MedSAM/results/raw/lora_student_fp32_cross_dataset_isic_2018_task1_validation_threshold_sweep_best_threshold.csv --best-s

In [7]:
# CROSS_DATASET_VALIDATION_SELECTED_RESULTS_CELL
from pathlib import Path
import pandas as pd
import sys

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))

VALIDATION_SELECTED_MODEL_SPECS = [
    {"label": "lora_student_fp32", "fixed_threshold_tag": "t05", "fixed_threshold": 0.5},
    {"label": "lora_student_qat_seed3", "fixed_threshold_tag": "t0625", "fixed_threshold": 0.625},
]
VALIDATION_SELECTED_DATASETS = ["isic_2018_task1", "busi"]

def read_first_row(path):
    if not path.exists():
        return None
    frame = pd.read_csv(path)
    if frame.empty:
        return None
    row = frame.iloc[0].to_dict()
    row["source"] = path.name
    return row

comparison_rows = []
for model_spec in VALIDATION_SELECTED_MODEL_SPECS:
    model_name = f"{model_spec['label']}_cross_dataset"
    for dataset_name in VALIDATION_SELECTED_DATASETS:
        fixed_path = REPO_ROOT / "results" / "tables" / f"{model_name}_{dataset_name}_test_{model_spec['fixed_threshold_tag']}_summary.csv"
        validation_sweep_path = REPO_ROOT / "results" / "tables" / f"{model_name}_{dataset_name}_validation_threshold_sweep.csv"
        validation_best_summary_path = REPO_ROOT / "results" / "tables" / f"{model_name}_{dataset_name}_validation_threshold_sweep_best_threshold_summary.csv"
        selected_test_path = REPO_ROOT / "results" / "tables" / f"{model_name}_{dataset_name}_test_validation_selected_threshold_summary.csv"
        oracle_test_path = REPO_ROOT / "results" / "tables" / f"{model_name}_{dataset_name}_test_threshold_sweep_best_threshold_summary.csv"

        fixed_row = read_first_row(fixed_path)
        validation_sweep_row = read_first_row(validation_sweep_path)
        validation_best_row = read_first_row(validation_best_summary_path)
        selected_test_row = read_first_row(selected_test_path)
        oracle_test_row = read_first_row(oracle_test_path)

        selected_threshold = validation_sweep_row.get("threshold") if validation_sweep_row else None
        oracle_threshold = oracle_test_row.get("selected_threshold") if oracle_test_row else None
        fixed_dice = fixed_row.get("mean_dice") if fixed_row else None
        selected_test_dice = selected_test_row.get("mean_dice") if selected_test_row else None
        oracle_test_dice = oracle_test_row.get("mean_dice") if oracle_test_row else None

        comparison_rows.append({
            "model": model_spec["label"],
            "dataset": dataset_name,
            "fixed_threshold": model_spec["fixed_threshold"],
            "fixed_test_dice": fixed_dice,
            "validation_selected_threshold": selected_threshold,
            "validation_best_dice": validation_sweep_row.get("mean_dice") if validation_sweep_row else None,
            "validation_selected_test_dice": selected_test_dice,
            "test_oracle_threshold": oracle_threshold,
            "test_oracle_dice": oracle_test_dice,
            "selected_minus_fixed_dice": None if fixed_dice is None or selected_test_dice is None else selected_test_dice - fixed_dice,
            "oracle_minus_selected_dice": None if oracle_test_dice is None or selected_test_dice is None else oracle_test_dice - selected_test_dice,
            "validation_selected_test_recall": selected_test_row.get("mean_recall") if selected_test_row else None,
            "validation_selected_test_precision": selected_test_row.get("mean_precision") if selected_test_row else None,
            "validation_selected_roi_ece": selected_test_row.get("mean_binary_ece_roi") if selected_test_row else None,
            "validation_selected_boundary_ece": selected_test_row.get("mean_binary_ece_boundary") if selected_test_row else None,
            "validation_selected_test_source": selected_test_path.name,
        })

comparison = pd.DataFrame(comparison_rows)
display(comparison)

available_selected_summaries = []
for table_path in sorted((REPO_ROOT / "results" / "tables").glob("*_cross_dataset_*_test_validation_selected_threshold_summary.csv")):
    row = read_first_row(table_path)
    if row:
        available_selected_summaries.append(row)

if available_selected_summaries:
    selected_summaries = pd.DataFrame(available_selected_summaries)
    selected_columns = [
        "source", "dataset", "split", "model_id", "sample_count", "mean_dice", "mean_iou",
        "mean_precision", "mean_recall", "mean_binary_ece_roi", "mean_binary_ece_boundary",
        "mean_latency_ms", "source_checkpoint",
    ]
    display(selected_summaries[[column for column in selected_columns if column in selected_summaries.columns]])
else:
    print("No validation-selected test summaries found yet. Run the previous cell first.")


,model,dataset,fixed_threshold,fixed_test_dice,validation_selected_threshold,validation_best_dice,validation_selected_test_dice,test_oracle_threshold,test_oracle_dice,selected_minus_fixed_dice,oracle_minus_selected_dice,validation_selected_test_recall,validation_selected_test_precision,validation_selected_roi_ece,validation_selected_boundary_ece,validation_selected_test_source
0,lora_student_fp32,isic_2018_task1,0.500,0.104154,0.025,0.342662,0.347377,0.025,0.347377,0.243223,0.000000,0.369639,0.539979,0.315096,0.222078,lora_student_fp32_cross_dataset_isic_2018_task...
1,lora_student_fp32,busi,0.500,0.059221,0.025,0.141133,0.112094,0.950,0.121894,0.052873,0.009800,0.534106,0.087647,0.080860,0.031515,lora_student_fp32_cross_dataset_busi_test_vali...
2,lora_student_qat_seed3,isic_2018_task1,0.625,0.099850,0.025,0.387948,0.390794,0.025,0.390794,0.290944,0.000000,0.377638,0.576648,0.298225,0.223608,lora_student_qat_seed3_cross_dataset_isic_2018...
3,lora_student_qat_seed3,busi,0.625,0.069855,0.025,0.124935,0.090835,0.925,0.129469,0.020980,0.038634,0.411069,0.080698,0.077285,0.043377,lora_student_qat_seed3_cross_dataset_busi_test...


,source,dataset,split,model_id,sample_count,mean_dice,mean_iou,mean_precision,mean_recall,mean_binary_ece_roi,mean_binary_ece_boundary,mean_latency_ms,source_checkpoint
0,lora_student_fp32_cross_dataset_busi_test_vali...,busi,test,lora_student_fp32_cross_dataset,117,0.112094,0.063567,0.087647,0.534106,0.080860,0.031515,34.358214,/mnt/e/automations/Deployable MedSAM/results/m...
1,lora_student_fp32_cross_dataset_isic_2018_task...,isic_2018_task1,test,lora_student_fp32_cross_dataset,1000,0.347377,0.238481,0.539979,0.369639,0.315096,0.222078,34.315448,/mnt/e/automations/Deployable MedSAM/results/m...
2,lora_student_qat_seed3_cross_dataset_busi_test...,busi,test,lora_student_qat_seed3_cross_dataset,117,0.090835,0.050574,0.080698,0.411069,0.077285,0.043377,31.778641,/mnt/e/automations/Deployable MedSAM/results/m...
3,lora_student_qat_seed3_cross_dataset_isic_2018...,isic_2018_task1,test,lora_student_qat_seed3_cross_dataset,1000,0.390794,0.270752,0.576648,0.377638,0.298225,0.223608,32.942888,/mnt/e/automations/Deployable MedSAM/results/m...
